# Lattice A/B Generator Notebook v5 — Type A Descriptor-LHS Sampler + Model Reload/CAD Export + Slice Images

Type A는 `N개 후보 생성 → Start/End 기반 구조인자 추출 → 구조인자 공간 LHS로 M개 선정 → 선정본만 STL/STP export` 방식으로 수정한 버전입니다.


추가: COM/SOUND용 DLP slice image 생성, 6개 모델 병합, template idx/gcode/preview 복사, 5-layer prepend 후처리 셀을 포함합니다.

## 1. Import + Settings

In [1]:
# -*- coding: utf-8 -*-
"""
Lattice A/B Generator Notebook v5 — Descriptor-LHS Sampler + Model Reload/CAD Export + Slice Images
=================================

Cell structure
--------------
1) Import + Settings
2) Common functions: graph, Type A, Type B, VF/radius, Start-End structural factors, export/STL
3) Runtime reset
4) Type A generation & Start-End structural factor extraction
5) Type B generation & Start-End structural factor extraction
6) Candidate/selected model reload for CAD export
7) Final save / preview
8) Slice image utilities
9) COM Type slice generation
10) SOUND Type slice generation
11) COM/SOUND 6-region merge + 5-layer postprocess

Main revision
-------------
- Export folder is automatically created as:
  C:/Users/김민겸/Desktop/AI-lattice architecture/Result_YYYYMMDD_HHMMSS
- Summary factors are computed from the final Start/End coordinate table, not only from the internal graph.
- Summary headers follow the uploaded Node-Strut extraction code naming/order.
- Slice image generation and postprocessing are added for COM/SOUND DLP workflows.
"""

from __future__ import annotations

import csv
import math
import re
import sys
import random
import statistics
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.stats import qmc
from scipy.optimize import linear_sum_assignment
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

try:
    import trimesh
    HAS_TRIMESH = True
except Exception:
    trimesh = None
    HAS_TRIMESH = False

try:
    from skimage import measure as sk_measure
    HAS_SKIMAGE = True
except Exception:
    sk_measure = None
    HAS_SKIMAGE = False

try:
    from IPython.display import display
except Exception:
    display = print


# =============================================================================
# User-editable defaults
# =============================================================================

EXPORT_BASE_DIR = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture")
DEFAULT_TYPE_B_IMPORT_PATH = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture\Type B Import\Variables.xlsx")

TEMPLATE_COLUMNS = [
    "Start-x", "Start-y", "Start-z", "Start-Radius",
    "End-x", "End-y", "End-z", "End-radius",
]

# Uploaded Node-Strut extraction code summary header names/order.
START_END_SUMMARY_COLUMNS = [
    "모델명",
    "Type",
    "Source",
    "Target VF",
    "Actual VF",
    "Node 개수",
    "Strut 개수",
    "Global: Strut No. at Nodes-AVG",
    "Global: Strut No. at Nodes-STDEV",
    "Global: Length - AVG",
    "Global: Length - STDEV",
    "Global: Angle-Z (weighted with length)-AVG",
    "Global: Angle-Z (weighted with length)-STDEV",
    "Node: Length - AVG",
    "Node: Length - STDEV",
    "Node: Angle-Z (weighted with length)-AVG",
    "Node: Angle-Z (weighted with length)-STDEV",
    "Global: l/d AVG",
    "Global: l/d STDEV",
    "Node: l/d AVG",
    "Node: l/d STDEV",
    "Global: Angle-X (weighted with length)-AVG",
    "Global: Angle-X (weighted with length)-STDEV",
    "Node: Angle-X (weighted with length)-AVG",
    "Node: Angle-X (weighted with length)-STDEV",
    "Global: Angle-Y (weighted with length)-AVG",
    "Global: Angle-Y (weighted with length)-STDEV",
    "Node: Angle-Y (weighted with length)-AVG",
    "Node: Angle-Y (weighted with length)-STDEV",
]

EXTRA_SUMMARY_COLUMNS = [
    "connected_components",
    "maxwell_index_M_3D",
    "total_strut_length_mm",
    "mean_radius_mm",
    "std_radius_mm",
    "mean_diameter_mm",
    "surface_area_mm2",
    "surface_to_solid_volume_1_per_mm",
    "solid_volume_mm3_approx",
    "bbox_x_mm",
    "bbox_y_mm",
    "bbox_z_mm",
    "mass_center_x_mm",
    "mass_center_y_mm",
    "mass_center_z_mm",
]


# Descriptor-space LHS selection uses these columns by default.
# VF별 stratified selection을 쓰면 Target VF/Actual VF는 selection feature에서 제외하는 것이 안정적입니다.
DEFAULT_LHS_DESCRIPTOR_COLUMNS = [
    "Global: Strut No. at Nodes-AVG",
    "Global: Strut No. at Nodes-STDEV",
    "Global: Length - AVG",
    "Global: Length - STDEV",
    "Global: Angle-Z (weighted with length)-AVG",
    "Global: Angle-X (weighted with length)-AVG",
    "Global: Angle-Y (weighted with length)-AVG",
    "Global: l/d AVG",
    "Node: l/d AVG",
    "maxwell_index_M_3D",
    "surface_to_solid_volume_1_per_mm",
]


def make_result_output_dir(base_dir: Path = EXPORT_BASE_DIR) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return Path(base_dir) / f"Result_{timestamp}"


@dataclass
class GenerationConfig:
    total_length_mm: float = 30.0
    cells_per_axis: int = 5
    target_vfs: Tuple[float, ...] = (0.30, 0.45, 0.60)
    # Type A 2-stage sampler
    type_a_candidate_n: int = 60000
    type_a_selected_m: int = 150
    type_a_lhs_stratify_by_vf: bool = True
    save_candidate_start_end_excel: bool = True
    candidate_start_end_max_rows_per_xlsx: int = 900_000

    # Legacy counts used only if you manually run the old direct-generation loop.
    n_type_a_per_vf: int = 10
    n_type_b_per_vf: int = 10
    enable_type_a: bool = True
    enable_type_b: bool = False
    seed: int = 42
    output_dir: Path = None
    type_b_import_path: Path = DEFAULT_TYPE_B_IMPORT_PATH

    # Type A topology ranges
    type_a_random_interior_nodes_min: int = 4
    type_a_random_interior_nodes_max: int = 12
    type_a_random_face_pair_nodes_min: int = 1
    type_a_random_face_pair_nodes_max: int = 4
    type_a_min_degree: int = 3
    type_a_max_degree: int = 6

    # Type B modification ranges. Set jitter=0 and edge_dropout=0 to preserve imported graph.
    type_b_node_jitter_mm: float = 0.10
    type_b_edge_dropout_prob: float = 0.00
    type_b_anisotropic_scale_std: float = 0.015

    # STL/STP settings
    export_stl: bool = True
    export_stp: bool = True
    freecad_lib_path: Optional[str] = None  # 예: r"C:\Program Files\FreeCAD 0.21\bin"
    mesh_mode: str = "cylinders"  # "cylinders" or "sdf"
    cylinder_sections: int = 16
    node_sphere_subdivisions: int = 2
    sdf_resolution: int = 96

    # Numeric tolerance
    merge_round_digits: int = 6
    min_edge_length_mm: float = 0.05

    def __post_init__(self):
        if self.output_dir is None:
            self.output_dir = make_result_output_dir(EXPORT_BASE_DIR)

    @property
    def cell_size_mm(self) -> float:
        return self.total_length_mm / self.cells_per_axis


@dataclass
class LatticeModel:
    nodes: np.ndarray       # shape: (N, 3), centered coordinates, mm
    edges: np.ndarray       # shape: (S, 2), node indices
    radii: np.ndarray       # shape: (S,), strut radius, mm
    source_type: str        # "A" or "B"
    target_vf: float
    actual_vf: float = math.nan
    source_name: str = ""


# =============================================================================
# Notebook settings: 여기만 수정하면 전체 생성 조건이 바뀝니다.
# =============================================================================

CFG = GenerationConfig(
    total_length_mm=30.0,
    cells_per_axis=5,
    target_vfs=(0.30, 0.45, 0.60),
    type_a_candidate_n=60000,
    type_a_selected_m=150,
    type_a_lhs_stratify_by_vf=True,
    save_candidate_start_end_excel=True,
    candidate_start_end_max_rows_per_xlsx=900_000,
    n_type_a_per_vf=10,
    n_type_b_per_vf=10,
    enable_type_a=True,
    enable_type_b=False,
    seed=42,
    output_dir=make_result_output_dir(EXPORT_BASE_DIR),
    type_b_import_path=DEFAULT_TYPE_B_IMPORT_PATH,
    export_stl=True,
    export_stp=True,
    freecad_lib_path=None,       # FreeCAD import가 안 되면 r"C:\Program Files\FreeCAD 0.21\bin"처럼 지정
    mesh_mode="cylinders",      # "cylinders" or "sdf"
    cylinder_sections=16,
    sdf_resolution=96,
    type_b_node_jitter_mm=0.10,
    type_b_edge_dropout_prob=0.00,
)

# 셀별 실행 스위치
RUN_TYPE_A = CFG.enable_type_a
RUN_TYPE_B = CFG.enable_type_b
SAVE_AFTER_EACH_GENERATION_CELL = True

print("Configuration loaded — v5")
print(f"Total size      : {CFG.total_length_mm} mm")
print(f"Cell size       : {CFG.cell_size_mm} mm")
print(f"Cells           : {CFG.cells_per_axis} x {CFG.cells_per_axis} x {CFG.cells_per_axis}")
print(f"Target VFs      : {CFG.target_vfs}")
print(f"Type A enabled  : {RUN_TYPE_A}")
print(f"Type A candidates N : {CFG.type_a_candidate_n}")
print(f"Type A selected M   : {CFG.type_a_selected_m}")
print(f"Type A LHS by VF    : {CFG.type_a_lhs_stratify_by_vf}")
print(f"Type B enabled  : {RUN_TYPE_B}, n/VF={CFG.n_type_b_per_vf}")
print(f"Export base     : {EXPORT_BASE_DIR}")
print(f"Output dir      : {CFG.output_dir}")
print(f"Type B import   : {CFG.type_b_import_path}")


Configuration loaded — v5
Total size      : 30.0 mm
Cell size       : 6.0 mm
Cells           : 5 x 5 x 5
Target VFs      : (0.3, 0.45, 0.6)
Type A enabled  : True
Type A candidates N : 60000
Type A selected M   : 150
Type A LHS by VF    : True
Type B enabled  : False, n/VF=10
Export base     : C:\Users\김민겸\Desktop\AI-lattice architecture
Output dir      : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
Type B import   : C:\Users\김민겸\Desktop\AI-lattice architecture\Type B Import\Variables.xlsx


## 2. Common functions
Graph utility, Type A/B 생성, 5×5×5 확장, VF radius, Start/End 구조인자, Excel/STL export를 한 셀로 묶었습니다.

In [2]:
# 1. Basic graph utilities
# =============================================================================

def _key3(pt: Sequence[float], digits: int = 6) -> Tuple[float, float, float]:
    return tuple(np.round(np.asarray(pt, dtype=float), digits))


def deduplicate_nodes_edges(
    nodes: np.ndarray,
    edges: Iterable[Tuple[int, int]],
    digits: int = 6,
    min_edge_length: float = 0.05,
) -> Tuple[np.ndarray, np.ndarray]:
    """Merge identical/near-identical nodes and remove degenerate/duplicate edges."""
    node_map: Dict[Tuple[float, float, float], int] = {}
    new_nodes: List[np.ndarray] = []
    old_to_new: Dict[int, int] = {}

    for idx, p in enumerate(np.asarray(nodes, dtype=float)):
        k = _key3(p, digits)
        if k not in node_map:
            node_map[k] = len(new_nodes)
            new_nodes.append(np.asarray(p, dtype=float))
        old_to_new[idx] = node_map[k]

    new_nodes_arr = np.asarray(new_nodes, dtype=float)
    edge_set = set()
    for i, j in edges:
        a = old_to_new[int(i)]
        b = old_to_new[int(j)]
        if a == b:
            continue
        if np.linalg.norm(new_nodes_arr[a] - new_nodes_arr[b]) < min_edge_length:
            continue
        edge_set.add(tuple(sorted((a, b))))

    if not edge_set:
        return new_nodes_arr, np.zeros((0, 2), dtype=int)
    return new_nodes_arr, np.asarray(sorted(edge_set), dtype=int)


def connected_components(n_nodes: int, edges: np.ndarray) -> List[List[int]]:
    parent = list(range(n_nodes))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for a, b in np.asarray(edges, dtype=int):
        union(int(a), int(b))

    comps: Dict[int, List[int]] = {}
    for i in range(n_nodes):
        comps.setdefault(find(i), []).append(i)
    return list(comps.values())


def ensure_connected(nodes: np.ndarray, edges: np.ndarray, max_components_to_repair: int = 50) -> np.ndarray:
    """Bridge disconnected components by adding nearest inter-component edges.

    For imported Type B geometry, a sheet may contain many already-discretized
    short segments rather than true graph struts. In that case forcing thousands
    of components into one graph is both slow and physically misleading, so the
    function preserves the imported topology when the component count is too high.
    """
    if len(nodes) == 0:
        return edges
    edge_list = [tuple(map(int, e)) for e in np.asarray(edges, dtype=int)]
    if not edge_list:
        return np.zeros((0, 2), dtype=int)

    comps0 = connected_components(len(nodes), np.asarray(edge_list, dtype=int))
    if len(comps0) <= 1:
        return np.asarray(sorted(set(tuple(sorted(e)) for e in edge_list)), dtype=int)
    if len(comps0) > max_components_to_repair:
        return np.asarray(sorted(set(tuple(sorted(e)) for e in edge_list)), dtype=int)

    used = {tuple(sorted(e)) for e in edge_list}
    while True:
        comps = connected_components(len(nodes), np.asarray(edge_list, dtype=int))
        if len(comps) <= 1:
            break
        D = squareform(pdist(nodes)) if len(nodes) > 1 else np.zeros((1, 1))
        best_pair = None
        best_dist = math.inf
        for ca in range(len(comps)):
            for cb in range(ca + 1, len(comps)):
                for i in comps[ca]:
                    for j in comps[cb]:
                        d = D[i, j]
                        pair = tuple(sorted((int(i), int(j))))
                        if pair not in used and d < best_dist:
                            best_dist = float(d)
                            best_pair = pair
        if best_pair is None:
            break
        edge_list.append(best_pair)
        used.add(best_pair)

    return np.asarray(sorted(set(tuple(sorted(e)) for e in edge_list)), dtype=int)


# =============================================================================


# 2. Type A: new 1-cell node/strut graph generation
# =============================================================================

def _base_unit_nodes(L: float) -> List[List[float]]:
    """Stable seed nodes for one unit cell: corners, face centers, body center."""
    nodes: List[List[float]] = []

    # Corners
    for x in (0.0, L):
        for y in (0.0, L):
            for z in (0.0, L):
                nodes.append([x, y, z])

    # Face centers
    h = L / 2.0
    nodes += [
        [0.0, h, h], [L, h, h],
        [h, 0.0, h], [h, L, h],
        [h, h, 0.0], [h, h, L],
    ]

    # Body center
    nodes.append([h, h, h])
    return nodes


def generate_type_a_unit_cell(cfg: GenerationConfig, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate a one-cell lattice graph.

    Coordinate system: [0, cell_size]^3.
    The graph is later tiled to 5x5x5 and recentered to [-15, 15]^3.
    """
    L = cfg.cell_size_mm
    nodes: List[List[float]] = _base_unit_nodes(L)

    # Random interior nodes
    n_int = int(rng.integers(cfg.type_a_random_interior_nodes_min,
                             cfg.type_a_random_interior_nodes_max + 1))
    margin = 0.12 * L
    for _ in range(n_int):
        p = rng.uniform(margin, L - margin, size=3)
        nodes.append(p.tolist())

    # Periodic paired face nodes: same in-plane coordinate on opposite faces.
    n_face_pairs = int(rng.integers(cfg.type_a_random_face_pair_nodes_min,
                                    cfg.type_a_random_face_pair_nodes_max + 1))
    for _ in range(n_face_pairs):
        axis = int(rng.integers(0, 3))
        uv = rng.uniform(0.18 * L, 0.82 * L, size=2)
        p0 = np.zeros(3)
        p1 = np.zeros(3)
        other_axes = [a for a in range(3) if a != axis]
        p0[axis] = 0.0
        p1[axis] = L
        p0[other_axes] = uv
        p1[other_axes] = uv
        nodes.append(p0.tolist())
        nodes.append(p1.tolist())

    nodes_arr, _ = deduplicate_nodes_edges(np.asarray(nodes, dtype=float), [], cfg.merge_round_digits)

    # Distance-based graph selection
    D = squareform(pdist(nodes_arr))
    np.fill_diagonal(D, np.inf)
    n = len(nodes_arr)
    deg = np.zeros(n, dtype=int)
    edges = set()

    min_len = max(0.18 * L, cfg.min_edge_length_mm)
    max_len = 1.15 * L
    target_degree = rng.integers(cfg.type_a_min_degree, cfg.type_a_max_degree + 1, size=n)

    # First pass: each node connects to nearby nodes until minimum degree is satisfied.
    random_penalty = rng.uniform(0.0, 0.18 * L, size=D.shape)
    order_matrix = D + random_penalty
    for i in rng.permutation(n):
        order = np.argsort(order_matrix[i])
        for j in order:
            if deg[i] >= target_degree[i]:
                break
            if deg[j] >= cfg.type_a_max_degree:
                continue
            d = D[i, j]
            if not (min_len <= d <= max_len):
                continue
            pair = tuple(sorted((int(i), int(j))))
            if pair in edges:
                continue
            edges.add(pair)
            deg[i] += 1
            deg[j] += 1

    # Second pass: add a few diagonal/longer edges for mechanical path diversity.
    candidate_pairs: List[Tuple[float, Tuple[int, int]]] = []
    for i in range(n):
        for j in range(i + 1, n):
            d = D[i, j]
            if 0.45 * L <= d <= math.sqrt(3) * L:
                candidate_pairs.append((float(d + rng.uniform(0, 0.35 * L)), (i, j)))
    candidate_pairs.sort(key=lambda x: x[0])
    for _, pair in candidate_pairs:
        if rng.random() > 0.12:
            continue
        i, j = pair
        if deg[i] >= cfg.type_a_max_degree or deg[j] >= cfg.type_a_max_degree:
            continue
        edges.add(pair)
        deg[i] += 1
        deg[j] += 1

    nodes_arr, edges_arr = deduplicate_nodes_edges(nodes_arr, edges, cfg.merge_round_digits, cfg.min_edge_length_mm)
    edges_arr = ensure_connected(nodes_arr, edges_arr)
    return nodes_arr, edges_arr


# =============================================================================


# 3. 1-cell to 5x5x5 expansion
# =============================================================================

def standardize_unit_cell(nodes: np.ndarray, cell_size: float) -> np.ndarray:
    """Shift/scale imported unit-cell coordinates to [0, cell_size]^3."""
    nodes = np.asarray(nodes, dtype=float).copy()
    mins = nodes.min(axis=0)
    maxs = nodes.max(axis=0)
    spans = maxs - mins
    max_span = float(np.max(spans))
    if max_span <= 1e-9:
        raise ValueError("Cannot standardize a unit cell with zero coordinate span.")
    nodes = nodes - mins
    nodes *= cell_size / max_span
    return np.clip(nodes, 0.0, cell_size)


def tile_unit_cell(
    unit_nodes: np.ndarray,
    unit_edges: np.ndarray,
    cfg: GenerationConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    """Tile one unit cell to cfg.cells_per_axis^3 cells and merge boundary nodes."""
    unit_nodes = standardize_unit_cell(unit_nodes, cfg.cell_size_mm)
    Lc = cfg.cell_size_mm
    all_nodes: List[np.ndarray] = []
    all_edges: List[Tuple[int, int]] = []
    node_map: Dict[Tuple[float, float, float], int] = {}

    def add_node(p: np.ndarray) -> int:
        k = _key3(p, cfg.merge_round_digits)
        if k not in node_map:
            node_map[k] = len(all_nodes)
            all_nodes.append(np.asarray(p, dtype=float))
        return node_map[k]

    c = cfg.cells_per_axis
    for ix in range(c):
        for iy in range(c):
            for iz in range(c):
                shift = np.array([ix * Lc, iy * Lc, iz * Lc], dtype=float)
                local_to_global = [add_node(p + shift) for p in unit_nodes]
                for a, b in unit_edges:
                    ia = local_to_global[int(a)]
                    ib = local_to_global[int(b)]
                    if ia != ib:
                        all_edges.append(tuple(sorted((ia, ib))))

    nodes, edges = deduplicate_nodes_edges(
        np.asarray(all_nodes, dtype=float),
        all_edges,
        cfg.merge_round_digits,
        cfg.min_edge_length_mm,
    )

    # Recenter from [0, total_length] to [-total_length/2, total_length/2]
    nodes = nodes - cfg.total_length_mm / 2.0
    edges = ensure_connected(nodes, edges)
    return nodes, edges


# =============================================================================


def _is_template_header(row_values: Sequence[object]) -> bool:
    normalized = [str(v).strip() if v is not None else "" for v in row_values[:8]]
    return normalized == TEMPLATE_COLUMNS


def _to_float_or_none(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    try:
        s = str(v).strip().replace(",", "")
        if s == "" or s.startswith("="):
            return None
        return float(s)
    except Exception:
        return None


def import_type_b_lattices(path: Path, cfg: GenerationConfig) -> List[Tuple[str, LatticeModel]]:
    """
    Import all valid sheets from either one .xlsx file or every .xlsx file in a folder.

    Supported input formats
    -----------------------
    1) Variables template row 1:
       Start-x, Start-y, Start-z, Start-Radius, End-x, End-y, End-z, End-radius

    2) Uploaded Node-Strut style sheet:
       row 3 has x,y,z / S-x,S-y,S-z / E-x,E-y,E-z.
       Radius is not required because Type B is later re-radiused to the target VF.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Type B import path does not exist: {path}")

    files = [path] if path.is_file() else sorted([p for p in path.glob("*.xlsx") if not p.name.startswith("~$")])
    imported: List[Tuple[str, LatticeModel]] = []

    def build_lat_from_rows(fpath: Path, sheet_name: str, rows: List[Tuple[float, float, float, float, float, float, float, float]]):
        point_to_id: Dict[Tuple[float, float, float], int] = {}
        nodes: List[List[float]] = []
        edges: List[Tuple[int, int]] = []
        radii: List[float] = []

        def node_id(p: Sequence[float]) -> int:
            k = _key3(p, cfg.merge_round_digits)
            if k not in point_to_id:
                point_to_id[k] = len(nodes)
                nodes.append([float(p[0]), float(p[1]), float(p[2])])
            return point_to_id[k]

        for sx, sy, sz, sr, ex, ey, ez, er in rows:
            a = node_id((sx, sy, sz))
            b = node_id((ex, ey, ez))
            if a == b:
                continue
            edges.append(tuple(sorted((a, b))))
            radii.append(max(1e-6, 0.5 * (float(sr) + float(er))))

        if len(nodes) < 2 or len(edges) < 1:
            return None
        nodes_arr, edges_arr = deduplicate_nodes_edges(
            np.asarray(nodes, dtype=float), edges,
            cfg.merge_round_digits, cfg.min_edge_length_mm,
        )
        r0 = float(np.mean(radii)) if radii else 0.5
        return LatticeModel(
            nodes=nodes_arr,
            edges=edges_arr,
            radii=np.full(len(edges_arr), r0, dtype=float),
            source_type="B",
            target_vf=math.nan,
            source_name=f"{fpath.stem}:{sheet_name}",
        )

    for fpath in files:
        wb = load_workbook(fpath, data_only=True, read_only=True)
        try:
            for ws in wb.worksheets:
                rows = []
                header = [ws.cell(row=1, column=i).value for i in range(1, 9)]

                if _is_template_header(header):
                    for row in ws.iter_rows(min_row=2, values_only=True):
                        if row is None or all(v is None for v in row[:8]):
                            continue
                        vals = [_to_float_or_none(v) for v in row[:8]]
                        if any(v is None for v in vals):
                            continue
                        rows.append(tuple(vals))

                else:
                    # Node-Strut style: row 3 contains x,y,z,S-x,S-y,S-z,E-x,E-y,E-z.
                    h3 = [str(ws.cell(row=3, column=i).value).strip() if ws.cell(row=3, column=i).value is not None else "" for i in range(1, 14)]
                    looks_node_strut = (len(h3) >= 10 and h3[1:4] == ["x", "y", "z"] and h3[4:7] == ["S-x", "S-y", "S-z"] and h3[7:10] == ["E-x", "E-y", "E-z"])
                    if looks_node_strut:
                        for r in range(4, ws.max_row + 1):
                            sx = _to_float_or_none(ws.cell(r, 5).value)
                            sy = _to_float_or_none(ws.cell(r, 6).value)
                            sz = _to_float_or_none(ws.cell(r, 7).value)
                            ex = _to_float_or_none(ws.cell(r, 8).value)
                            ey = _to_float_or_none(ws.cell(r, 9).value)
                            ez = _to_float_or_none(ws.cell(r, 10).value)
                            if any(v is None for v in (sx, sy, sz, ex, ey, ez)):
                                continue
                            # placeholder radius; assigned again by target VF later
                            rows.append((sx, sy, sz, 0.5, ex, ey, ez, 0.5))

                lat = build_lat_from_rows(fpath, ws.title, rows)
                if lat is not None:
                    imported.append((lat.source_name, lat))
        finally:
            wb.close()
    return imported


def normalize_or_tile_imported_lattice(lat: LatticeModel, cfg: GenerationConfig) -> Tuple[np.ndarray, np.ndarray]:
    """
    If imported geometry looks like a 1-cell model, tile it.
    If it looks like a full 30 mm model, scale/recenter it to total_length_mm.
    """
    nodes = np.asarray(lat.nodes, dtype=float).copy()
    edges = np.asarray(lat.edges, dtype=int).copy()
    span = nodes.max(axis=0) - nodes.min(axis=0)
    max_span = float(np.max(span))

    # Heuristic: unit-cell if the largest span is close to or below one cell size.
    if max_span <= cfg.cell_size_mm * 1.35:
        return tile_unit_cell(nodes, edges, cfg)

    # Full-size source: uniformly scale max span to total length and recenter.
    mins = nodes.min(axis=0)
    center = 0.5 * (nodes.max(axis=0) + mins)
    nodes = nodes - center
    scale = cfg.total_length_mm / max_span if max_span > 1e-9 else 1.0
    nodes *= scale
    # Keep the lattice inside the bounding cube after scaling.
    half = cfg.total_length_mm / 2.0
    nodes = np.clip(nodes, -half, half)
    nodes, edges = deduplicate_nodes_edges(nodes, edges, cfg.merge_round_digits, cfg.min_edge_length_mm)
    edges = ensure_connected(nodes, edges)
    return nodes, edges


def modify_type_b_lattice(nodes: np.ndarray, edges: np.ndarray, cfg: GenerationConfig,
                          rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    """Apply mild geometry variation while preserving the imported lattice's identity."""
    nodes = np.asarray(nodes, dtype=float).copy()
    edges = np.asarray(edges, dtype=int).copy()
    half = cfg.total_length_mm / 2.0

    # Keep boundary nodes stable enough for dimensional accuracy.
    on_boundary = np.any(np.isclose(np.abs(nodes), half, atol=1e-6), axis=1)
    if cfg.type_b_node_jitter_mm > 0:
        jitter = rng.normal(0.0, cfg.type_b_node_jitter_mm, size=nodes.shape)
        jitter[on_boundary] *= 0.25
        nodes += jitter

    if cfg.type_b_anisotropic_scale_std > 0:
        scale = rng.normal(1.0, cfg.type_b_anisotropic_scale_std, size=3)
        nodes *= scale

    nodes = np.clip(nodes, -half, half)

    if cfg.type_b_edge_dropout_prob > 0 and len(edges) > 4:
        keep = rng.random(len(edges)) >= cfg.type_b_edge_dropout_prob
        if keep.sum() >= 4:
            edges = edges[keep]

    nodes, edges = deduplicate_nodes_edges(nodes, edges, cfg.merge_round_digits, cfg.min_edge_length_mm)
    edges = ensure_connected(nodes, edges)
    return nodes, edges




# =============================================================================
# VF/radius assignment + Start/End based structural factor extraction + Excel export
# =============================================================================

def strut_lengths(nodes: np.ndarray, edges: np.ndarray) -> np.ndarray:
    p0 = nodes[edges[:, 0]]
    p1 = nodes[edges[:, 1]]
    return np.linalg.norm(p1 - p0, axis=1)


def assign_uniform_radius_for_vf(nodes: np.ndarray, edges: np.ndarray, target_vf: float,
                                 total_length: float) -> Tuple[np.ndarray, float]:
    """
    VF approximation:
    VF ≈ sum(pi*r^2*L_i) / total_length^3
    r  = sqrt(VF * total_length^3 / (pi * sum(L_i)))
    """
    lengths = strut_lengths(nodes, edges)
    total_len = float(np.sum(lengths))
    if total_len <= 1e-12:
        raise ValueError("Cannot assign radius because total strut length is zero.")
    r = math.sqrt(max(target_vf * total_length**3 / (math.pi * total_len), 1e-12))
    radii = np.full(len(edges), r, dtype=float)
    actual_vf = calculate_volume_fraction(nodes, edges, radii, total_length)
    return radii, actual_vf


def calculate_volume_fraction(nodes: np.ndarray, edges: np.ndarray, radii: np.ndarray,
                              total_length: float) -> float:
    lengths = strut_lengths(nodes, edges)
    volume = float(np.sum(math.pi * radii**2 * lengths))
    return volume / (total_length**3)


def model_rows(lat: LatticeModel) -> List[List[float]]:
    rows: List[List[float]] = []
    for (a, b), r in zip(lat.edges, lat.radii):
        p0 = lat.nodes[int(a)]
        p1 = lat.nodes[int(b)]
        rows.append([
            float(p0[0]), float(p0[1]), float(p0[2]), float(r),
            float(p1[0]), float(p1[1]), float(p1[2]), float(r),
        ])
    return rows


def model_variables_df(lat: LatticeModel) -> pd.DataFrame:
    return pd.DataFrame(model_rows(lat), columns=TEMPLATE_COLUMNS)



def safe_filename(text: object, max_len: int = 120) -> str:
    """Windows-safe filename from model ID/label."""
    s = str(text).strip()
    s = re.sub(r'[\\/:*?"<>|]+', '_', s)
    s = re.sub(r'\s+', '_', s)
    s = s.strip('._')
    return s[:max_len] if len(s) > max_len else s


def make_model_label(
    model_id: object,
    source_type: str = "A",
    target_vf: Optional[float] = None,
    source_stage: str = "Candidate",
    candidate_id: Optional[object] = None,
    selected_id: Optional[object] = None,
) -> str:
    """Human-readable label for later search/reload/CAD export.

    Examples
    --------
    - A_Candidate_C00001_VF030
    - A_Selected_S001_from_C00001_VF030
    """
    stype = str(source_type).upper() if source_type else "X"
    stage = str(source_stage).replace(" ", "_")
    vf_tag = "VFNA"
    try:
        if target_vf is not None and not pd.isna(target_vf):
            vf_tag = f"VF{int(round(float(target_vf) * 100)):03d}"
    except Exception:
        vf_tag = "VFNA"

    if selected_id is not None and candidate_id is not None:
        return f"{stype}_Selected_S{int(selected_id):03d}_from_{candidate_id}_{vf_tag}"
    return f"{stype}_{stage}_{model_id}_{vf_tag}"


def export_model_registry(summary_rows: List[Dict[str, object]], out_xlsx: Path, out_csv: Optional[Path] = None) -> pd.DataFrame:
    """Save compact registry for later model search/reload.

    The registry is the file to open first when choosing models manually.
    It contains both machine-friendly IDs and human-readable labels.
    """
    df = pd.DataFrame(summary_rows)
    if df.empty:
        return df
    preferred = [
        "Model_Label", "모델명", "Candidate_ID", "Selected_Model_ID", "Type", "Source",
        "Target VF", "Actual VF", "LHS_Selected", "Selection_Rank", "LHS_Distance",
        "Global: Strut No. at Nodes-AVG", "Global: Length - AVG", "Global: Length - STDEV",
        "Global: Angle-Z (weighted with length)-AVG", "Global: Angle-X (weighted with length)-AVG",
        "Global: Angle-Y (weighted with length)-AVG", "Global: l/d AVG", "Node: l/d AVG",
        "maxwell_index_M_3D", "surface_to_solid_volume_1_per_mm",
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    df = df[cols]
    out_xlsx.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(out_xlsx, index=False)
    if out_csv is not None:
        df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return df


def _round_key3(x, y, z, nd=6):
    return (round(float(x), nd), round(float(y), nd), round(float(z), nd))


def _canonical_pair(k1, k2):
    return tuple(sorted([k1, k2]))


def _mean(values):
    vals = [float(v) for v in values if v is not None and math.isfinite(float(v))]
    return statistics.mean(vals) if vals else math.nan


def _stdev_sample(values):
    vals = [float(v) for v in values if v is not None and math.isfinite(float(v))]
    if len(vals) <= 1:
        return 0.0 if len(vals) == 1 else math.nan
    return statistics.stdev(vals)


def _weighted_mean(values, weights):
    pairs = [(float(v), float(w)) for v, w in zip(values, weights)
             if v is not None and w is not None and math.isfinite(float(v)) and math.isfinite(float(w)) and float(w) > 0]
    if not pairs:
        return math.nan
    sw = sum(w for _, w in pairs)
    return sum(v * w for v, w in pairs) / sw if sw > 0 else math.nan


def _weighted_stdev(values, weights):
    pairs = [(float(v), float(w)) for v, w in zip(values, weights)
             if v is not None and w is not None and math.isfinite(float(v)) and math.isfinite(float(w)) and float(w) > 0]
    if len(pairs) <= 1:
        return 0.0 if len(pairs) == 1 else math.nan
    sw = sum(w for _, w in pairs)
    if sw <= 0:
        return math.nan
    wm = sum(v * w for v, w in pairs) / sw
    return math.sqrt(sum(w * (v - wm) ** 2 for v, w in pairs) / sw)


def _axis_angles_from_start_end(p1, p2):
    """
    Angle definition matched to uploaded Node-Strut extraction code:
    - Angle-Z = asin(|dz| / L): angle contribution relative to XY plane
    - Angle-X = asin(|dx| / L)
    - Angle-Y = asin(|dy| / L)
    """
    dx = float(p1[0]) - float(p2[0])
    dy = float(p1[1]) - float(p2[1])
    dz = float(p1[2]) - float(p2[2])
    L = math.sqrt(dx * dx + dy * dy + dz * dz)
    if L <= 1e-12:
        return 0.0, math.nan, math.nan, math.nan

    def a(delta):
        ratio = max(0.0, min(1.0, abs(float(delta)) / L))
        return math.degrees(math.asin(ratio))

    return L, a(dz), a(dx), a(dy)


def extract_start_end_structural_factors(model_id: str, lat: LatticeModel, cfg: GenerationConfig) -> Dict[str, object]:
    """Compute structural factors directly from final Start/End coordinates and radii."""
    df = model_variables_df(lat)
    node_to_struts = defaultdict(list)
    point_set = set()
    struts = []
    unique = {}

    for _, row in df.iterrows():
        sx, sy, sz = float(row["Start-x"]), float(row["Start-y"]), float(row["Start-z"])
        ex, ey, ez = float(row["End-x"]), float(row["End-y"]), float(row["End-z"])
        sr, er = float(row["Start-Radius"]), float(row["End-radius"])
        p1 = (sx, sy, sz)
        p2 = (ex, ey, ez)
        L, angle_z, angle_x, angle_y = _axis_angles_from_start_end(p1, p2)
        if L <= cfg.min_edge_length_mm:
            continue
        k1 = _round_key3(sx, sy, sz, cfg.merge_round_digits)
        k2 = _round_key3(ex, ey, ez, cfg.merge_round_digits)
        pair_key = _canonical_pair(k1, k2)
        radius = max(1e-12, 0.5 * (sr + er))
        diameter = 2.0 * radius
        l_over_d = L / diameter
        item = {
            "k1": k1, "k2": k2, "unique_key": pair_key,
            "length": L,
            "radius": radius,
            "diameter": diameter,
            "l_over_d": l_over_d,
            "angle_z": angle_z,
            "angle_x": angle_x,
            "angle_y": angle_y,
            "p1": p1,
            "p2": p2,
        }
        struts.append(item)
        node_to_struts[k1].append(item)
        node_to_struts[k2].append(item)
        point_set.add(k1)
        point_set.add(k2)
        if pair_key not in unique:
            unique[pair_key] = item

    unique_struts = list(unique.values())
    nodes = sorted(point_set)
    lengths = [s["length"] for s in unique_struts]
    radii = [s["radius"] for s in unique_struts]
    diameters = [s["diameter"] for s in unique_struts]
    all_l_over_d = [s["l_over_d"] for s in struts]

    degree_values = [len(node_to_struts[k]) for k in nodes]

    def global_angle_stats(angle_key):
        vals = [s[angle_key] for s in unique_struts]
        w = [s["length"] for s in unique_struts]
        return _weighted_mean(vals, w), _weighted_stdev(vals, w)

    def node_length_stats():
        node_avgs = []
        for k in nodes:
            vals = [s["length"] for s in node_to_struts[k]]
            if vals:
                node_avgs.append(_mean(vals))
        return _mean(node_avgs), _stdev_sample(node_avgs)

    def node_l_over_d_stats():
        node_avgs = []
        for k in nodes:
            vals = [s["l_over_d"] for s in node_to_struts[k]]
            if vals:
                node_avgs.append(_mean(vals))
        return _mean(node_avgs), _stdev_sample(node_avgs)

    def node_angle_stats(angle_key):
        node_weighted = []
        for k in nodes:
            connected = node_to_struts[k]
            vals = [s[angle_key] for s in connected]
            w = [s["length"] for s in connected]
            if vals:
                node_weighted.append(_weighted_mean(vals, w))
        return _mean(node_weighted), _stdev_sample(node_weighted)

    g_az_avg, g_az_std = global_angle_stats("angle_z")
    n_az_avg, n_az_std = node_angle_stats("angle_z")
    g_ax_avg, g_ax_std = global_angle_stats("angle_x")
    n_ax_avg, n_ax_std = node_angle_stats("angle_x")
    g_ay_avg, g_ay_std = global_angle_stats("angle_y")
    n_ay_avg, n_ay_std = node_angle_stats("angle_y")
    n_len_avg, n_len_std = node_length_stats()
    n_ld_avg, n_ld_std = node_l_over_d_stats()

    # Additional descriptors retained from the previous notebook.
    node_arr = np.asarray(list(nodes), dtype=float) if nodes else np.zeros((0, 3))
    p0 = np.asarray([s["p1"] for s in unique_struts], dtype=float) if unique_struts else np.zeros((0, 3))
    p1 = np.asarray([s["p2"] for s in unique_struts], dtype=float) if unique_struts else np.zeros((0, 3))
    mid = 0.5 * (p0 + p1) if len(unique_struts) else np.zeros((0, 3))
    solid_volume_terms = [math.pi * s["radius"] ** 2 * s["length"] for s in unique_struts]
    surface_terms = [2.0 * math.pi * s["radius"] * s["length"] for s in unique_struts]
    solid_volume = float(sum(solid_volume_terms))
    surface_area = float(sum(surface_terms))
    mass_w = np.asarray(solid_volume_terms, dtype=float)
    if mass_w.sum() > 0:
        mass_w = mass_w / mass_w.sum()
        mc = np.sum(mid * mass_w[:, None], axis=0)
    else:
        mc = np.array([math.nan, math.nan, math.nan])

    # connected component count based on internal graph remains useful for topology check.
    comps = connected_components(len(lat.nodes), lat.edges) if len(lat.nodes) else []

    row = {
        "모델명": model_id,
        "Type": lat.source_type,
        "Source": lat.source_name,
        "Target VF": lat.target_vf,
        "Actual VF": lat.actual_vf,
        "Node 개수": len(nodes),
        "Strut 개수": len(unique_struts),
        "Global: Strut No. at Nodes-AVG": _mean(degree_values),
        "Global: Strut No. at Nodes-STDEV": _stdev_sample(degree_values),
        "Global: Length - AVG": _mean(lengths),
        "Global: Length - STDEV": _stdev_sample(lengths),
        "Global: Angle-Z (weighted with length)-AVG": g_az_avg,
        "Global: Angle-Z (weighted with length)-STDEV": g_az_std,
        "Node: Length - AVG": n_len_avg,
        "Node: Length - STDEV": n_len_std,
        "Node: Angle-Z (weighted with length)-AVG": n_az_avg,
        "Node: Angle-Z (weighted with length)-STDEV": n_az_std,
        "Global: l/d AVG": _mean(all_l_over_d),
        "Global: l/d STDEV": _stdev_sample(all_l_over_d),
        "Node: l/d AVG": n_ld_avg,
        "Node: l/d STDEV": n_ld_std,
        "Global: Angle-X (weighted with length)-AVG": g_ax_avg,
        "Global: Angle-X (weighted with length)-STDEV": g_ax_std,
        "Node: Angle-X (weighted with length)-AVG": n_ax_avg,
        "Node: Angle-X (weighted with length)-STDEV": n_ax_std,
        "Global: Angle-Y (weighted with length)-AVG": g_ay_avg,
        "Global: Angle-Y (weighted with length)-STDEV": g_ay_std,
        "Node: Angle-Y (weighted with length)-AVG": n_ay_avg,
        "Node: Angle-Y (weighted with length)-STDEV": n_ay_std,
        "connected_components": int(len(comps)),
        "maxwell_index_M_3D": int(len(unique_struts) - 3 * len(nodes) + 6),
        "total_strut_length_mm": float(sum(lengths)) if lengths else math.nan,
        "mean_radius_mm": _mean(radii),
        "std_radius_mm": _stdev_sample(radii),
        "mean_diameter_mm": _mean(diameters),
        "surface_area_mm2": surface_area,
        "surface_to_solid_volume_1_per_mm": surface_area / solid_volume if solid_volume > 0 else math.nan,
        "solid_volume_mm3_approx": solid_volume,
        "bbox_x_mm": float(node_arr[:, 0].max() - node_arr[:, 0].min()) if len(node_arr) else 0,
        "bbox_y_mm": float(node_arr[:, 1].max() - node_arr[:, 1].min()) if len(node_arr) else 0,
        "bbox_z_mm": float(node_arr[:, 2].max() - node_arr[:, 2].min()) if len(node_arr) else 0,
        "mass_center_x_mm": float(mc[0]),
        "mass_center_y_mm": float(mc[1]),
        "mass_center_z_mm": float(mc[2]),
    }
    return row


def style_worksheet(ws, n_cols: int = 8) -> None:
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(bold=True, color="FFFFFF")
    thin = Side(style="thin", color="D9E2F3")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border
    ws.freeze_panes = "A2"

    for col_idx in range(1, n_cols + 1):
        letter = get_column_letter(col_idx)
        ws.column_dimensions[letter].width = 16
        for cell in ws[letter]:
            cell.border = border
            if cell.row > 1:
                cell.number_format = "0.000000"


def export_variables_workbook(models: Dict[str, LatticeModel], out_path: Path) -> None:
    wb = Workbook()
    default = wb.active
    wb.remove(default)

    for model_id, lat in models.items():
        ws = wb.create_sheet(title=str(model_id))
        ws.append(TEMPLATE_COLUMNS)
        for row in model_rows(lat):
            ws.append(row)
        style_worksheet(ws, 8)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    wb.save(out_path)


def export_summary(summary_rows: List[Dict[str, object]], xlsx_path: Path, csv_path: Path) -> None:
    if not summary_rows:
        return
    headers = [h for h in START_END_SUMMARY_COLUMNS + EXTRA_SUMMARY_COLUMNS if any(h in row for row in summary_rows)]
    for row in summary_rows:
        for k in row.keys():
            if k not in headers:
                headers.append(k)

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(summary_rows)

    wb = Workbook()
    ws = wb.active
    ws.title = "Summary"
    ws.append(headers)
    for row in summary_rows:
        ws.append([row.get(h, "") for h in headers])

    header_fill = PatternFill("solid", fgColor="375623")
    header_font = Font(bold=True, color="FFFFFF")
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    for idx, h in enumerate(headers, start=1):
        width = min(max(len(str(h)) + 2, 12), 38)
        ws.column_dimensions[get_column_letter(idx)].width = width
        for cell in ws[get_column_letter(idx)]:
            if isinstance(cell.value, float):
                cell.number_format = "0.000000"
    xlsx_path.parent.mkdir(parents=True, exist_ok=True)
    wb.save(xlsx_path)


# =============================================================================
# 7. STL export
# =============================================================================

def _cylinder_between(p0: np.ndarray, p1: np.ndarray, radius: float, sections: int):
    if not HAS_TRIMESH:
        raise ImportError("trimesh is required for STL export. Run: pip install trimesh")
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    vec = p1 - p0
    height = float(np.linalg.norm(vec))
    if height <= 1e-9 or radius <= 1e-12:
        return None
    mesh = trimesh.creation.cylinder(radius=float(radius), height=height, sections=sections)
    transform = trimesh.geometry.align_vectors(np.array([0.0, 0.0, 1.0]), vec / height)
    transform[:3, 3] = 0.5 * (p0 + p1)
    mesh.apply_transform(transform)
    return mesh


def build_cylinder_sphere_mesh(lat: LatticeModel, cfg: GenerationConfig):
    if not HAS_TRIMESH:
        raise ImportError("trimesh is required for STL export. Run: pip install trimesh")

    meshes = []
    node_r = np.zeros(len(lat.nodes), dtype=float)
    for e_idx, (a, b) in enumerate(lat.edges):
        r = float(lat.radii[e_idx])
        node_r[int(a)] = max(node_r[int(a)], r)
        node_r[int(b)] = max(node_r[int(b)], r)
        cyl = _cylinder_between(lat.nodes[int(a)], lat.nodes[int(b)], r, cfg.cylinder_sections)
        if cyl is not None:
            meshes.append(cyl)

    for n_idx, p in enumerate(lat.nodes):
        r = max(float(node_r[n_idx]), 1e-6)
        sph = trimesh.creation.icosphere(subdivisions=cfg.node_sphere_subdivisions, radius=r)
        sph.apply_translation(np.asarray(p, dtype=float))
        meshes.append(sph)

    if not meshes:
        raise ValueError("No mesh primitives were generated.")
    return trimesh.util.concatenate(meshes)


def build_sdf_mesh(lat: LatticeModel, cfg: GenerationConfig):
    """
    Slower but more fusion-like mesh via capsule SDF + marching cubes.
    Good for preview/fused-STL but not exact CAD boolean geometry.
    """
    if not HAS_SKIMAGE:
        raise ImportError("scikit-image is required for SDF STL. Run: pip install scikit-image")
    if not HAS_TRIMESH:
        raise ImportError("trimesh is required for STL export. Run: pip install trimesh")

    half = cfg.total_length_mm / 2.0
    pad = cfg.total_length_mm * 0.03
    res = int(cfg.sdf_resolution)
    x = np.linspace(-half - pad, half + pad, res)
    y = np.linspace(-half - pad, half + pad, res)
    z = np.linspace(-half - pad, half + pad, res)
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
    pts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    sdf = np.full(len(pts), np.inf, dtype=float)

    for (a, b), r in zip(lat.edges, lat.radii):
        p0 = lat.nodes[int(a)]
        p1 = lat.nodes[int(b)]
        ba = p1 - p0
        denom = float(np.dot(ba, ba))
        if denom <= 1e-12:
            continue
        pa = pts - p0
        h = np.clip(np.dot(pa, ba) / denom, 0.0, 1.0)
        closest = p0 + np.outer(h, ba)
        sdf = np.minimum(sdf, np.linalg.norm(pts - closest, axis=1) - float(r))

    # Node spheres to cap intersections.
    node_r = np.zeros(len(lat.nodes), dtype=float)
    for (a, b), r in zip(lat.edges, lat.radii):
        node_r[int(a)] = max(node_r[int(a)], float(r))
        node_r[int(b)] = max(node_r[int(b)], float(r))
    for p, r in zip(lat.nodes, node_r):
        sdf = np.minimum(sdf, np.linalg.norm(pts - p, axis=1) - float(r))

    # Clip to bounding cube.
    box_sdf = np.max(np.abs(pts) - half, axis=1)
    sdf = np.maximum(sdf, box_sdf)
    sdf_vol = sdf.reshape(X.shape)
    spacing = (x[1] - x[0], y[1] - y[0], z[1] - z[0])
    verts, faces, _, _ = sk_measure.marching_cubes(sdf_vol, level=0.0, spacing=spacing)
    verts += np.array([-half - pad, -half - pad, -half - pad])
    return trimesh.Trimesh(vertices=verts, faces=faces, process=True)


def export_stl(lat: LatticeModel, out_path: Path, cfg: GenerationConfig) -> None:
    if cfg.mesh_mode.lower() == "sdf":
        mesh = build_sdf_mesh(lat, cfg)
    else:
        mesh = build_cylinder_sphere_mesh(lat, cfg)
    mesh.export(out_path)


def _try_import_freecad(cfg: GenerationConfig):
    """Try to import FreeCAD/Part for STEP export."""
    if cfg.freecad_lib_path:
        p = str(cfg.freecad_lib_path)
        if p not in sys.path:
            sys.path.append(p)
    try:
        import FreeCAD
        import Part
        from FreeCAD import Base
        return FreeCAD, Part, Base
    except Exception:
        return None, None, None


def export_stp(lat: LatticeModel, out_path: Path, cfg: GenerationConfig) -> bool:
    """Export lattice as STEP/STP using FreeCAD if available.

    The STEP is exported as a compound of analytical cylinders and node spheres.
    Boolean union is intentionally skipped for speed/stability with many stochastic lattices.
    """
    FreeCAD, Part, Base = _try_import_freecad(cfg)
    if FreeCAD is None or Part is None:
        return False

    out_path.parent.mkdir(parents=True, exist_ok=True)
    doc = None
    try:
        doc = FreeCAD.newDocument("lattice_step_export")
        shapes = []

        node_r = np.zeros(len(lat.nodes), dtype=float)
        for e_idx, (a, b) in enumerate(lat.edges):
            r = float(lat.radii[e_idx])
            node_r[int(a)] = max(node_r[int(a)], r)
            node_r[int(b)] = max(node_r[int(b)], r)

        for e_idx, (a, b) in enumerate(lat.edges):
            p0 = np.asarray(lat.nodes[int(a)], dtype=float)
            p1 = np.asarray(lat.nodes[int(b)], dtype=float)
            vec = p1 - p0
            h = float(np.linalg.norm(vec))
            r = float(lat.radii[e_idx])
            if h <= cfg.min_edge_length_mm or r <= 0:
                continue
            base = Base.Vector(float(p0[0]), float(p0[1]), float(p0[2]))
            direction = Base.Vector(float(vec[0]), float(vec[1]), float(vec[2]))
            cyl = Part.makeCylinder(r, h, base, direction)
            shapes.append(cyl)

        for i, p in enumerate(lat.nodes):
            r = float(max(node_r[i], np.mean(lat.radii) if len(lat.radii) else 0.05))
            if r <= 0:
                continue
            sph = Part.makeSphere(r, Base.Vector(float(p[0]), float(p[1]), float(p[2])))
            shapes.append(sph)

        if not shapes:
            return False
        comp = Part.Compound(shapes)
        comp.exportStep(str(out_path))
        return True
    except Exception as e:
        print(f"[WARN] STP export failed for {out_path.name}: {e}")
        return False
    finally:
        try:
            if doc is not None:
                FreeCAD.closeDocument(doc.Name)
        except Exception:
            pass


# =============================================================================


# =============================================================================

# =============================================================================
# 8. Type A candidate pool + descriptor-space LHS selection
# =============================================================================

def _balanced_vf_schedule(target_vfs: Sequence[float], n_total: int) -> List[float]:
    """Return a balanced VF schedule of length n_total."""
    vfs = [float(v) for v in target_vfs]
    if not vfs:
        raise ValueError("target_vfs is empty")
    reps = math.ceil(n_total / len(vfs))
    schedule = (vfs * reps)[:n_total]
    return schedule


def export_variables_long_table_chunked(
    model_dict: Dict[str, LatticeModel],
    out_prefix: Path,
    max_rows_per_xlsx: int = 900_000,
) -> List[Path]:
    """Save candidate Start/End data as long-table Excel files.

    For N=5000, one-sheet-per-model Excel is impractical. This function writes
    long-table files with Model_ID columns and splits them before Excel's row limit.
    """
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    part = 1
    current_rows = 0

    def new_workbook():
        wb = Workbook(write_only=True)
        ws = wb.create_sheet("Start_End_LongTable")
        ws.append(["Model_ID", "Model_Label", "Type", "Target VF", "Actual VF"] + TEMPLATE_COLUMNS)
        return wb, ws

    wb, ws = new_workbook()
    current_rows = 1

    for model_id, lat in model_dict.items():
        rows = model_rows(lat)
        for row in rows:
            if current_rows >= max_rows_per_xlsx:
                path = out_prefix.parent / f"{out_prefix.name}_Part{part:03d}.xlsx"
                wb.save(path)
                paths.append(path)
                part += 1
                wb, ws = new_workbook()
                current_rows = 1
            model_label = make_model_label(model_id, lat.source_type, lat.target_vf, source_stage="Candidate")
            ws.append([model_id, model_label, lat.source_type, lat.target_vf, lat.actual_vf] + row)
            current_rows += 1

    path = out_prefix.parent / f"{out_prefix.name}_Part{part:03d}.xlsx"
    wb.save(path)
    paths.append(path)
    return paths


def _valid_descriptor_columns(df: pd.DataFrame, requested: Sequence[str]) -> List[str]:
    cols = []
    for c in requested:
        if c in df.columns:
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().sum() >= 2 and vals.nunique(dropna=True) >= 2:
                cols.append(c)
    return cols


def _robust_normalize_descriptor_matrix(df: pd.DataFrame, feature_cols: Sequence[str]) -> Tuple[np.ndarray, List[str]]:
    cols = _valid_descriptor_columns(df, feature_cols)
    if len(cols) == 0:
        raise ValueError("No valid descriptor columns are available for LHS selection.")
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    q05 = X.quantile(0.05)
    q95 = X.quantile(0.95)
    denom = (q95 - q05).replace(0, np.nan)
    Xn = (X - q05) / denom
    Xn = Xn.replace([np.inf, -np.inf], np.nan).fillna(0.5)
    Xn = Xn.clip(0.0, 1.0)
    return Xn.to_numpy(dtype=float), cols


def lhs_select_candidate_ids(
    summary_df: pd.DataFrame,
    n_select: int,
    cfg: GenerationConfig,
    feature_cols: Sequence[str] = DEFAULT_LHS_DESCRIPTOR_COLUMNS,
) -> Tuple[List[str], pd.DataFrame]:
    """Select n_select candidate IDs using LHS targets in descriptor space.

    If cfg.type_a_lhs_stratify_by_vf=True, selection is balanced across Target VF groups.
    """
    if len(summary_df) == 0:
        return [], summary_df
    if n_select >= len(summary_df):
        out = summary_df.copy()
        out["LHS_Selected"] = True
        out["Selection_Rank"] = np.arange(1, len(out) + 1)
        out["LHS_Distance"] = 0.0
        return out["모델명"].astype(str).tolist(), out

    rng_local = np.random.default_rng(cfg.seed + 2026)
    selected: List[str] = []
    annotated_parts = []

    def select_within_group(group_df: pd.DataFrame, m: int) -> pd.DataFrame:
        g = group_df.copy().reset_index(drop=True)
        if m <= 0:
            g["LHS_Selected"] = False
            g["Selection_Rank"] = ""
            g["LHS_Distance"] = np.nan
            return g
        if m >= len(g):
            g["LHS_Selected"] = True
            g["Selection_Rank"] = np.arange(1, len(g) + 1)
            g["LHS_Distance"] = 0.0
            return g

        X, used_cols = _robust_normalize_descriptor_matrix(g, feature_cols)
        sampler = qmc.LatinHypercube(d=X.shape[1], seed=int(rng_local.integers(1, 2**31 - 1)))
        targets = sampler.random(n=m)
        D = cdist(targets, X, metric="euclidean")
        row_ind, col_ind = linear_sum_assignment(D)
        chosen_cols = col_ind[:m]
        chosen_target_rows = row_ind[:m]

        g["LHS_Selected"] = False
        g["Selection_Rank"] = ""
        g["LHS_Distance"] = np.nan
        for rank, (ti, ci) in enumerate(sorted(zip(chosen_target_rows, chosen_cols), key=lambda x: x[0]), start=1):
            g.loc[ci, "LHS_Selected"] = True
            g.loc[ci, "Selection_Rank"] = rank
            g.loc[ci, "LHS_Distance"] = float(D[ti, ci])
        g["LHS_Feature_Count"] = len(used_cols)
        g["LHS_Features"] = "; ".join(used_cols)
        return g

    if cfg.type_a_lhs_stratify_by_vf and "Target VF" in summary_df.columns:
        groups = [(vf, g.copy()) for vf, g in summary_df.groupby("Target VF", sort=True)]
        base = n_select // len(groups)
        rem = n_select % len(groups)
        for idx, (vf, g) in enumerate(groups):
            m = base + (1 if idx < rem else 0)
            annotated_parts.append(select_within_group(g, min(m, len(g))))
    else:
        annotated_parts.append(select_within_group(summary_df.copy(), n_select))

    annotated = pd.concat(annotated_parts, ignore_index=True)
    selected_ids = annotated.loc[annotated["LHS_Selected"].astype(bool), "모델명"].astype(str).tolist()

    # If small groups caused under-selection, fill remaining by farthest distance from selected points.
    if len(selected_ids) < min(n_select, len(summary_df)):
        remaining = annotated[~annotated["모델명"].astype(str).isin(selected_ids)].copy()
        need = min(n_select, len(summary_df)) - len(selected_ids)
        if need > 0 and len(remaining) > 0:
            X_all, used_cols = _robust_normalize_descriptor_matrix(annotated, feature_cols)
            ids_all = annotated["모델명"].astype(str).tolist()
            selected_idx = [ids_all.index(sid) for sid in selected_ids if sid in ids_all]
            if selected_idx:
                dmin = cdist(X_all, X_all[selected_idx]).min(axis=1)
            else:
                center = np.full((1, X_all.shape[1]), 0.5)
                dmin = cdist(X_all, center).ravel()
            annotated["_fill_distance"] = dmin
            fill_df = annotated[~annotated["모델명"].astype(str).isin(selected_ids)].sort_values("_fill_distance", ascending=False).head(need)
            fill_ids = fill_df["모델명"].astype(str).tolist()
            selected_ids.extend(fill_ids)
            annotated.loc[annotated["모델명"].astype(str).isin(fill_ids), "LHS_Selected"] = True
            annotated = annotated.drop(columns=["_fill_distance"], errors="ignore")

    return selected_ids[:n_select], annotated


def generate_type_a_candidate_pool(cfg: GenerationConfig, rng: np.random.Generator):
    candidate_models: Dict[str, LatticeModel] = {}
    candidate_summaries: List[Dict[str, object]] = []
    schedule = _balanced_vf_schedule(cfg.target_vfs, cfg.type_a_candidate_n)

    print(f"[Type A candidate generation] N={cfg.type_a_candidate_n}, target VFs={cfg.target_vfs}")
    for i, vf in enumerate(schedule, start=1):
        model_id = f"C{i:05d}"
        lat = create_type_a_model(float(vf), cfg, rng)
        lat.source_name = "candidate_pool_random_unit_cell"
        candidate_models[model_id] = lat
        summary = extract_start_end_structural_factors(model_id, lat, cfg)
        summary["Candidate_ID"] = model_id
        summary["Model_Label"] = make_model_label(model_id, lat.source_type, lat.target_vf, source_stage="Candidate")
        candidate_summaries.append(summary)
        if i == 1 or i % 100 == 0 or i == cfg.type_a_candidate_n:
            print(f"  generated {i:>5}/{cfg.type_a_candidate_n} | last={model_id}, target VF={vf:.2f}, actual VF={lat.actual_vf:.4f}")

    return candidate_models, candidate_summaries


def save_type_a_candidate_outputs(candidate_models, candidate_summaries, cfg: GenerationConfig):
    candidate_dir = cfg.output_dir / "TypeA_Candidate_Pool"
    candidate_dir.mkdir(parents=True, exist_ok=True)

    summary_xlsx = candidate_dir / "TypeA_Candidate_Structural_Factors.xlsx"
    summary_csv = candidate_dir / "TypeA_Candidate_Structural_Factors.csv"
    registry_xlsx = candidate_dir / "TypeA_Candidate_Model_Registry.xlsx"
    registry_csv = candidate_dir / "TypeA_Candidate_Model_Registry.csv"
    export_summary(candidate_summaries, summary_xlsx, summary_csv)
    export_model_registry(candidate_summaries, registry_xlsx, registry_csv)

    start_end_paths = []
    if cfg.save_candidate_start_end_excel:
        start_end_prefix = candidate_dir / "TypeA_Candidate_StartEnd_LongTable"
        start_end_paths = export_variables_long_table_chunked(
            candidate_models,
            start_end_prefix,
            max_rows_per_xlsx=cfg.candidate_start_end_max_rows_per_xlsx,
        )

    print("\nSaved Type A candidate pool outputs")
    print(f"  Candidate folder : {candidate_dir}")
    print(f"  Summary XLSX     : {summary_xlsx}")
    print(f"  Summary CSV      : {summary_csv}")
    print(f"  Model registry   : {registry_xlsx}")
    if start_end_paths:
        print(f"  Start/End XLSX   : {len(start_end_paths)} chunk file(s)")
        for p in start_end_paths[:5]:
            print(f"    - {p}")
        if len(start_end_paths) > 5:
            print("    ...")
    return candidate_dir, summary_xlsx, summary_csv, start_end_paths


def save_type_a_selected_lhs_outputs(candidate_models, candidate_summaries, selected_ids: List[str], annotated_df: pd.DataFrame, cfg: GenerationConfig):
    selected_dir = cfg.output_dir / "TypeA_Selected_LHS"
    selected_dir.mkdir(parents=True, exist_ok=True)
    stl_selected_dir = selected_dir / "STL"
    stp_selected_dir = selected_dir / "STP"
    if cfg.export_stl:
        stl_selected_dir.mkdir(parents=True, exist_ok=True)
    if cfg.export_stp:
        stp_selected_dir.mkdir(parents=True, exist_ok=True)

    selected_models: Dict[str, LatticeModel] = {}
    selected_summaries: List[Dict[str, object]] = []
    selected_id_map = {}

    annotated_lookup = annotated_df.set_index(annotated_df["모델명"].astype(str)).to_dict(orient="index")

    stp_available = True
    for new_idx, old_id in enumerate(selected_ids, start=1):
        new_id = str(new_idx)
        lat = candidate_models[str(old_id)]
        selected_models[new_id] = lat
        selected_id_map[new_id] = str(old_id)

        summary = extract_start_end_structural_factors(new_id, lat, cfg)
        summary["Candidate_ID"] = str(old_id)
        summary["Selected_Model_ID"] = new_id
        summary["Model_Label"] = make_model_label(new_id, lat.source_type, lat.target_vf, source_stage="Selected", candidate_id=old_id, selected_id=new_idx)
        old_meta = annotated_lookup.get(str(old_id), {})
        for k in ["LHS_Selected", "Selection_Rank", "LHS_Distance", "LHS_Feature_Count", "LHS_Features"]:
            if k in old_meta:
                summary[k] = old_meta[k]
        selected_summaries.append(summary)

        if cfg.export_stl:
            export_stl(lat, stl_selected_dir / f"{new_id}.stl", cfg)
        if cfg.export_stp:
            ok = export_stp(lat, stp_selected_dir / f"{new_id}.stp", cfg)
            if not ok:
                stp_available = False

    variables_path = selected_dir / "TypeA_Selected_StartEnd_Variables.xlsx"
    summary_xlsx = selected_dir / "TypeA_Selected_Structural_Factors.xlsx"
    summary_csv = selected_dir / "TypeA_Selected_Structural_Factors.csv"
    annotated_xlsx = selected_dir / "TypeA_Candidate_LHS_Annotated.xlsx"
    idmap_path = selected_dir / "TypeA_Selected_ID_Map.csv"
    registry_xlsx = selected_dir / "TypeA_Selected_Model_Registry.xlsx"
    registry_csv = selected_dir / "TypeA_Selected_Model_Registry.csv"

    export_variables_workbook(selected_models, variables_path)
    export_summary(selected_summaries, summary_xlsx, summary_csv)
    export_model_registry(selected_summaries, registry_xlsx, registry_csv)
    annotated_df.to_excel(annotated_xlsx, index=False)
    pd.DataFrame([{"Selected_Model_ID": k, "Candidate_ID": v} for k, v in selected_id_map.items()]).to_csv(idmap_path, index=False, encoding="utf-8-sig")

    print("\nSaved Type A selected LHS outputs")
    print(f"  Selected folder   : {selected_dir}")
    print(f"  Variables workbook: {variables_path}")
    print(f"  Summary workbook  : {summary_xlsx}")
    print(f"  Summary CSV       : {summary_csv}")
    print(f"  LHS annotated     : {annotated_xlsx}")
    print(f"  ID map            : {idmap_path}")
    print(f"  Model registry    : {registry_xlsx}")
    if cfg.export_stl:
        print(f"  STL folder        : {stl_selected_dir}")
    if cfg.export_stp:
        print(f"  STP folder        : {stp_selected_dir}")
        if not stp_available:
            print("  [WARN] STP export requires FreeCAD Python modules. Set CFG.freecad_lib_path or run inside a FreeCAD Python environment.")

    return selected_models, selected_summaries, selected_dir

# =============================================================================
# 9. Batch helpers + runtime state
# =============================================================================

def create_type_a_model(target_vf: float, cfg: GenerationConfig, rng: np.random.Generator) -> LatticeModel:
    unit_nodes, unit_edges = generate_type_a_unit_cell(cfg, rng)
    nodes, edges = tile_unit_cell(unit_nodes, unit_edges, cfg)
    radii, actual_vf = assign_uniform_radius_for_vf(nodes, edges, target_vf, cfg.total_length_mm)
    return LatticeModel(nodes=nodes, edges=edges, radii=radii,
                        source_type="A", target_vf=target_vf,
                        actual_vf=actual_vf, source_name="random_unit_cell")


def create_type_b_model(target_vf: float, base_lat: LatticeModel, cfg: GenerationConfig,
                        rng: np.random.Generator) -> LatticeModel:
    nodes, edges = normalize_or_tile_imported_lattice(base_lat, cfg)
    nodes, edges = modify_type_b_lattice(nodes, edges, cfg, rng)
    radii, actual_vf = assign_uniform_radius_for_vf(nodes, edges, target_vf, cfg.total_length_mm)
    return LatticeModel(nodes=nodes, edges=edges, radii=radii,
                        source_type="B", target_vf=target_vf,
                        actual_vf=actual_vf, source_name=base_lat.source_name)



# =============================================================================
# Model reload / manual CAD export helpers
# =============================================================================

def lattice_from_start_end_dataframe(
    df: pd.DataFrame,
    cfg: GenerationConfig = CFG,
    model_id: str = "manual",
    source_type: str = "A",
    target_vf: Optional[float] = None,
    source_name: str = "reloaded_start_end",
) -> LatticeModel:
    """Reconstruct LatticeModel from exported Start/End coordinate table.

    Works with both:
    - Candidate long-table filtered to one Model_ID/Model_Label
    - Selected model sheet from TypeA_Selected_StartEnd_Variables.xlsx
    """
    colmap = {str(c).strip().lower(): c for c in df.columns}

    def get_col(*names):
        for n in names:
            key = str(n).strip().lower()
            if key in colmap:
                return colmap[key]
        raise KeyError(f"Required column not found. Tried: {names}")

    sx = get_col("Start-x", "S-x", "Sx")
    sy = get_col("Start-y", "S-y", "Sy")
    sz = get_col("Start-z", "S-z", "Sz")
    sr = get_col("Start-Radius", "Start-radius", "S-radius", "S-Radius", "Radius")
    ex = get_col("End-x", "E-x", "Ex")
    ey = get_col("End-y", "E-y", "Ey")
    ez = get_col("End-z", "E-z", "Ez")
    er = get_col("End-radius", "End-Radius", "E-radius", "E-Radius", "Radius")

    node_index: Dict[Tuple[float, float, float], int] = {}
    nodes: List[List[float]] = []
    edge_radius: Dict[Tuple[int, int], List[float]] = defaultdict(list)

    def add_node(pt):
        key = _key3(pt, cfg.merge_round_digits)
        if key not in node_index:
            node_index[key] = len(nodes)
            nodes.append([float(pt[0]), float(pt[1]), float(pt[2])])
        return node_index[key]

    for _, row in df.iterrows():
        vals = [row.get(sx), row.get(sy), row.get(sz), row.get(sr), row.get(ex), row.get(ey), row.get(ez), row.get(er)]
        vals = [pd.to_numeric(v, errors="coerce") for v in vals]
        if any(pd.isna(v) for v in vals[:3] + vals[4:7]):
            continue
        p0 = np.array(vals[:3], dtype=float)
        p1 = np.array(vals[4:7], dtype=float)
        if np.linalg.norm(p1 - p0) <= cfg.min_edge_length_mm:
            continue
        r0 = float(vals[3]) if not pd.isna(vals[3]) else math.nan
        r1 = float(vals[7]) if not pd.isna(vals[7]) else r0
        r = np.nanmean([r0, r1])
        if not math.isfinite(float(r)) or float(r) <= 0:
            continue
        a = add_node(p0)
        b = add_node(p1)
        if a == b:
            continue
        edge = tuple(sorted((a, b)))
        edge_radius[edge].append(float(r))

    if not edge_radius:
        raise ValueError(f"No valid struts found for model {model_id}.")

    edges = np.array(list(edge_radius.keys()), dtype=int)
    radii = np.array([float(np.mean(v)) for v in edge_radius.values()], dtype=float)
    nodes_arr = np.array(nodes, dtype=float)

    P0 = nodes_arr[edges[:, 0]]
    P1 = nodes_arr[edges[:, 1]]
    lengths = np.linalg.norm(P1 - P0, axis=1)
    actual_vf = float(np.sum(math.pi * radii**2 * lengths) / (cfg.total_length_mm ** 3))

    if target_vf is None:
        for c in ["Target VF", "target_vf", "Target_VF"]:
            if c in df.columns:
                tv = pd.to_numeric(df[c], errors="coerce").dropna()
                if len(tv):
                    target_vf = float(tv.iloc[0])
                    break
    if target_vf is None:
        target_vf = actual_vf

    return LatticeModel(
        nodes=nodes_arr,
        edges=edges,
        radii=radii,
        source_type=source_type,
        target_vf=float(target_vf),
        actual_vf=actual_vf,
        source_name=source_name,
    )


def read_summary_registry(result_dir: Path, source: str = "candidate") -> pd.DataFrame:
    """Read compact registry/summary for model search."""
    result_dir = Path(result_dir)
    source = source.lower()
    if source.startswith("cand"):
        candidates = [
            result_dir / "TypeA_Candidate_Pool" / "TypeA_Candidate_Model_Registry.xlsx",
            result_dir / "TypeA_Candidate_Pool" / "TypeA_Candidate_Structural_Factors.xlsx",
        ]
    else:
        candidates = [
            result_dir / "TypeA_Selected_LHS" / "TypeA_Selected_Model_Registry.xlsx",
            result_dir / "TypeA_Selected_LHS" / "TypeA_Selected_Structural_Factors.xlsx",
        ]
    for p in candidates:
        if p.exists():
            return pd.read_excel(p)
    raise FileNotFoundError(f"No registry/summary file found in {result_dir} for source={source}")


def find_models_by_descriptor(
    result_dir: Path,
    source: str = "candidate",
    query: Optional[str] = None,
    sort_by: Optional[str] = None,
    ascending: bool = True,
    head: int = 30,
) -> pd.DataFrame:
    """Search model registry by pandas query/sort.

    Examples
    --------
    find_models_by_descriptor(result_dir, query='`Target VF` == 0.45', sort_by='Global: l/d AVG')
    """
    df = read_summary_registry(result_dir, source=source)
    out = df.copy()
    if query:
        out = out.query(query)
    if sort_by and sort_by in out.columns:
        out = out.sort_values(sort_by, ascending=ascending)
    keep = [
        "Model_Label", "모델명", "Candidate_ID", "Selected_Model_ID", "Target VF", "Actual VF",
        "Selection_Rank", "LHS_Distance", "Global: Length - AVG", "Global: l/d AVG",
        "Global: Angle-Z (weighted with length)-AVG", "maxwell_index_M_3D",
    ]
    keep = [c for c in keep if c in out.columns]
    return out[keep].head(head) if keep else out.head(head)


def load_type_a_candidate_models_from_longtable(
    result_dir: Path,
    ids_or_labels: Sequence[object],
    cfg: GenerationConfig = CFG,
) -> Dict[str, LatticeModel]:
    """Load candidate models by Candidate_ID or Model_Label from chunked long-table Excel files."""
    result_dir = Path(result_dir)
    ids = {str(x) for x in ids_or_labels}
    folder = result_dir / "TypeA_Candidate_Pool"
    files = sorted(folder.glob("TypeA_Candidate_StartEnd_LongTable_Part*.xlsx"))
    if not files:
        raise FileNotFoundError(f"No candidate long-table files found in {folder}")

    collected: Dict[str, List[pd.DataFrame]] = {}
    labels_seen: Dict[str, str] = {}
    for path in files:
        df = pd.read_excel(path)
        if "Model_ID" not in df.columns:
            continue
        mid = df["Model_ID"].astype(str)
        mask = mid.isin(ids)
        if "Model_Label" in df.columns:
            mask = mask | df["Model_Label"].astype(str).isin(ids)
        sub = df.loc[mask].copy()
        if sub.empty:
            continue
        for actual_id, g in sub.groupby(sub["Model_ID"].astype(str)):
            collected.setdefault(str(actual_id), []).append(g)
            if "Model_Label" in g.columns:
                labels_seen[str(actual_id)] = str(g["Model_Label"].iloc[0])

    loaded: Dict[str, LatticeModel] = {}
    for key, parts in collected.items():
        if not parts:
            continue
        g = pd.concat(parts, ignore_index=True)
        label = labels_seen.get(key, key)
        lat = lattice_from_start_end_dataframe(g, cfg, model_id=key, source_type="A", source_name=label)
        loaded[key] = lat
    if not loaded:
        print(f"[WARN] No candidate models loaded for {sorted(ids)}")
    return loaded


def load_type_a_selected_models_from_workbook(
    result_dir: Path,
    ids_or_labels: Sequence[object],
    cfg: GenerationConfig = CFG,
) -> Dict[str, LatticeModel]:
    """Load selected models by Selected_Model_ID, sheet name, Candidate_ID, or Model_Label."""
    result_dir = Path(result_dir)
    selected_dir = result_dir / "TypeA_Selected_LHS"
    variables_path = selected_dir / "TypeA_Selected_StartEnd_Variables.xlsx"
    if not variables_path.exists():
        raise FileNotFoundError(f"Selected Variables workbook not found: {variables_path}")

    ids = {str(x) for x in ids_or_labels}
    registry = None
    try:
        registry = read_summary_registry(result_dir, source="selected")
    except Exception:
        registry = None

    resolved_ids = set(ids)
    if registry is not None and not registry.empty:
        for _, row in registry.iterrows():
            sid = str(row.get("Selected_Model_ID", row.get("모델명", "")))
            aliases = {sid, str(row.get("모델명", "")), str(row.get("Candidate_ID", "")), str(row.get("Model_Label", ""))}
            if aliases & ids:
                resolved_ids.add(sid)

    xls = pd.ExcelFile(variables_path)
    loaded = {}
    for sheet in xls.sheet_names:
        if str(sheet) not in resolved_ids:
            continue
        df = pd.read_excel(variables_path, sheet_name=sheet)
        lat = lattice_from_start_end_dataframe(df, cfg, model_id=str(sheet), source_type="A", source_name=f"selected_{sheet}")
        loaded[str(sheet)] = lat
    if not loaded:
        print(f"[WARN] No selected models loaded for {sorted(ids)}")
    return loaded


def export_manual_loaded_models_to_cad(
    result_dir: Path,
    ids_or_labels: Sequence[object],
    source: str = "candidate",
    cfg: GenerationConfig = CFG,
    out_folder_name: Optional[str] = None,
    export_stl_flag: bool = True,
    export_stp_flag: bool = True,
) -> Tuple[Dict[str, LatticeModel], List[Dict[str, object]], Path]:
    """Reload selected model IDs/labels and export Variables + summary + STL/STP."""
    result_dir = Path(result_dir)
    if not ids_or_labels:
        raise ValueError("ids_or_labels is empty. Put Candidate_ID, Selected_Model_ID, or Model_Label values first.")

    if source.lower().startswith("cand"):
        loaded = load_type_a_candidate_models_from_longtable(result_dir, ids_or_labels, cfg)
        source_tag = "Candidate"
    else:
        loaded = load_type_a_selected_models_from_workbook(result_dir, ids_or_labels, cfg)
        source_tag = "Selected"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if out_folder_name is None:
        out_folder_name = f"Manual_Loaded_{source_tag}_CAD_{timestamp}"
    out_dir = result_dir / out_folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    export_models = {}
    summaries_out = []
    for idx, (mid, lat) in enumerate(loaded.items(), start=1):
        label = lat.source_name if lat.source_name else make_model_label(mid, lat.source_type, lat.target_vf, source_stage=source_tag)
        out_id = safe_filename(label)
        export_models[out_id] = lat
        s = extract_start_end_structural_factors(out_id, lat, cfg)
        s["Original_Model_ID"] = mid
        s["Model_Label"] = label
        summaries_out.append(s)

    variables_path = out_dir / "Manual_Loaded_StartEnd_Variables.xlsx"
    summary_xlsx = out_dir / "Manual_Loaded_Structural_Factors.xlsx"
    summary_csv = out_dir / "Manual_Loaded_Structural_Factors.csv"
    registry_xlsx = out_dir / "Manual_Loaded_Model_Registry.xlsx"
    registry_csv = out_dir / "Manual_Loaded_Model_Registry.csv"

    export_variables_workbook(export_models, variables_path)
    export_summary(summaries_out, summary_xlsx, summary_csv)
    export_model_registry(summaries_out, registry_xlsx, registry_csv)

    if export_stl_flag:
        stl_out = out_dir / "STL"
        stl_out.mkdir(parents=True, exist_ok=True)
        for mid, lat in export_models.items():
            export_stl(lat, stl_out / f"{safe_filename(mid)}.stl", cfg)

    stp_available = True
    if export_stp_flag:
        stp_out = out_dir / "STP"
        stp_out.mkdir(parents=True, exist_ok=True)
        for mid, lat in export_models.items():
            ok = export_stp(lat, stp_out / f"{safe_filename(mid)}.stp", cfg)
            if not ok:
                stp_available = False
        if not stp_available:
            print("[WARN] STP export requires FreeCAD Python modules. STL/Excel were still saved.")

    print("\nManual model reload/export finished")
    print(f"  Source          : {source_tag}")
    print(f"  Loaded models   : {len(export_models)}")
    print(f"  Output folder   : {out_dir}")
    print(f"  Variables XLSX  : {variables_path}")
    print(f"  Summary XLSX    : {summary_xlsx}")
    if export_stl_flag:
        print(f"  STL folder      : {out_dir / 'STL'}")
    if export_stp_flag:
        print(f"  STP folder      : {out_dir / 'STP'}")
    return export_models, summaries_out, out_dir

def reset_runtime_state(cfg: GenerationConfig = CFG):
    """모델 번호, summary, 출력 폴더 상태를 초기화합니다."""
    global rng, models, summaries, model_counter, stl_dir, stp_dir
    rng = np.random.default_rng(cfg.seed)
    random.seed(cfg.seed)
    models = {}
    summaries = []
    model_counter = 1
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    stl_dir = cfg.output_dir / "STL"
    stp_dir = cfg.output_dir / "STP"
    if cfg.export_stl:
        stl_dir.mkdir(parents=True, exist_ok=True)
    if cfg.export_stp:
        stp_dir.mkdir(parents=True, exist_ok=True)
    return models, summaries


def register_model(lat: LatticeModel, cfg: GenerationConfig = CFG):
    """모델명을 1, 2, 3... 순서로 부여하고 Start/End 기반 summary/STL을 저장 큐에 추가합니다."""
    global model_counter
    model_id = str(model_counter)
    models[model_id] = lat

    # 핵심 수정: 내부 edge 배열이 아니라 최종 Start/End table에서 구조인자 추출
    summary = extract_start_end_structural_factors(model_id, lat, cfg)
    summaries.append(summary)

    if cfg.export_stl:
        export_stl(lat, stl_dir / f"{model_id}.stl", cfg)
    if cfg.export_stp:
        ok = export_stp(lat, stp_dir / f"{model_id}.stp", cfg)
        if not ok:
            print("[WARN] STP export skipped: FreeCAD Python modules are not available.")

    model_counter += 1
    return model_id, summary


def save_current_outputs(cfg: GenerationConfig = CFG):
    """현재까지 생성된 모든 모델을 하나의 Variables workbook과 summary 파일로 저장합니다."""
    variables_path = cfg.output_dir / "Variables_All_Models.xlsx"
    summary_xlsx_path = cfg.output_dir / "Structural_Factors_Summary.xlsx"
    summary_csv_path = cfg.output_dir / "Structural_Factors_Summary.csv"

    export_variables_workbook(models, variables_path)
    export_summary(summaries, summary_xlsx_path, summary_csv_path)

    print("\nSaved current outputs")
    print(f"  Output folder      : {cfg.output_dir}")
    print(f"  Variables workbook : {variables_path}")
    print(f"  Summary workbook   : {summary_xlsx_path}")
    print(f"  Summary CSV        : {summary_csv_path}")
    if cfg.export_stl:
        print(f"  STL folder         : {cfg.output_dir / 'STL'}")
    if cfg.export_stp:
        print(f"  STP folder         : {cfg.output_dir / 'STP'}")
    return variables_path, summary_xlsx_path, summary_csv_path


def show_summary_tail(n=10):
    if not summaries:
        print("No models generated yet.")
        return None
    df = pd.DataFrame(summaries)
    display(df.tail(n))
    return df


## 3. Runtime reset

In [3]:
# =============================================================================
# Runtime reset
# =============================================================================
# settings cell을 다시 실행하면 Output folder가 새 Result_YYYYMMDD_HHMMSS로 바뀝니다.
reset_runtime_state(CFG)
print("Runtime state reset")
print("Next model id =", model_counter)
print("Output folder =", CFG.output_dir)


Runtime state reset
Next model id = 1
Output folder = C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242


## 4. Type A — N 후보 생성 → 구조인자 추출 → Descriptor-LHS M개 선정 → 선정본 STL/STP 저장

In [4]:
# =============================================================================
# Type A: Candidate pool + descriptor-space LHS selection
# =============================================================================
# 실행 결과
# 1) Result_*/TypeA_Candidate_Pool/
#    - TypeA_Candidate_Structural_Factors.xlsx / .csv
#    - TypeA_Candidate_StartEnd_LongTable_PartXXX.xlsx  (N개 후보, STL/STP 없음)
#
# 2) Result_*/TypeA_Selected_LHS/
#    - TypeA_Selected_StartEnd_Variables.xlsx           (M개 선정본, 모델별 sheet)
#    - TypeA_Selected_Structural_Factors.xlsx / .csv
#    - STL/*.stl                                       (M개 선정본만)
#    - STP/*.stp                                       (FreeCAD 사용 가능 시 M개 선정본만)
# =============================================================================

if RUN_TYPE_A:
    type_a_candidate_models, type_a_candidate_summaries = generate_type_a_candidate_pool(CFG, rng)

    # N개 후보의 구조인자 + Start/End 좌표/radius 저장. STL/STP는 저장하지 않음.
    type_a_candidate_dir, type_a_candidate_summary_xlsx, type_a_candidate_summary_csv, type_a_start_end_paths = save_type_a_candidate_outputs(
        type_a_candidate_models,
        type_a_candidate_summaries,
        CFG,
    )

    candidate_summary_df = pd.DataFrame(type_a_candidate_summaries)
    selected_candidate_ids, type_a_lhs_annotated_df = lhs_select_candidate_ids(
        candidate_summary_df,
        n_select=CFG.type_a_selected_m,
        cfg=CFG,
        feature_cols=DEFAULT_LHS_DESCRIPTOR_COLUMNS,
    )

    print(f"\n[LHS selection] selected {len(selected_candidate_ids)} / {len(type_a_candidate_models)} candidates")
    print("First selected candidate IDs:", selected_candidate_ids[:10])

    # 선정된 M개만 별도 폴더에 Excel + STL + STP 저장
    type_a_selected_models, type_a_selected_summaries, type_a_selected_dir = save_type_a_selected_lhs_outputs(
        type_a_candidate_models,
        type_a_candidate_summaries,
        selected_candidate_ids,
        type_a_lhs_annotated_df,
        CFG,
    )

    # Final preview cell과 Type B 후속 실행을 위해 selected Type A를 현재 runtime result로 등록
    models.clear()
    models.update(type_a_selected_models)
    summaries.clear()
    summaries.extend(type_a_selected_summaries)

    print("\nType A descriptor-LHS sampling finished.")
    display(pd.DataFrame(type_a_selected_summaries).head(10))
else:
    print("Type A generation is disabled. Set RUN_TYPE_A=True or CFG.enable_type_a=True to run it.")


[Type A candidate generation] N=60000, target VFs=(0.3, 0.45, 0.6)
  generated     1/60000 | last=C00001, target VF=0.30, actual VF=0.3000
  generated   100/60000 | last=C00100, target VF=0.30, actual VF=0.3000
  generated   200/60000 | last=C00200, target VF=0.45, actual VF=0.4500
  generated   300/60000 | last=C00300, target VF=0.60, actual VF=0.6000
  generated   400/60000 | last=C00400, target VF=0.30, actual VF=0.3000
  generated   500/60000 | last=C00500, target VF=0.45, actual VF=0.4500
  generated   600/60000 | last=C00600, target VF=0.60, actual VF=0.6000
  generated   700/60000 | last=C00700, target VF=0.30, actual VF=0.3000
  generated   800/60000 | last=C00800, target VF=0.45, actual VF=0.4500
  generated   900/60000 | last=C00900, target VF=0.60, actual VF=0.6000
  generated  1000/60000 | last=C01000, target VF=0.30, actual VF=0.3000
  generated  1100/60000 | last=C01100, target VF=0.45, actual VF=0.4500
  generated  1200/60000 | last=C01200, target VF=0.60, actual VF=0.60

,모델명,Type,Source,Target VF,Actual VF,Node 개수,Strut 개수,Global: Strut No. at Nodes-AVG,Global: Strut No. at Nodes-STDEV,Global: Length - AVG,...,mass_center_y_mm,mass_center_z_mm,Candidate_ID,Selected_Model_ID,Model_Label,LHS_Selected,Selection_Rank,LHS_Distance,LHS_Feature_Count,LHS_Features
0,1,A,candidate_pool_random_unit_cell,0.3,0.3,2441,9100,7.455961,4.758578,3.152885,...,-0.070272,0.071628,C01483,1,A_Selected_S001_from_C01483_VF030,True,46,0.679573,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
1,2,A,candidate_pool_random_unit_cell,0.3,0.3,2241,8950,7.987506,4.909614,3.174418,...,0.188997,0.438495,C01816,2,A_Selected_S002_from_C01816_VF030,True,47,0.642327,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
2,3,A,candidate_pool_random_unit_cell,0.3,0.3,1741,7000,8.041356,5.166927,3.516430,...,-0.320868,0.022946,C01921,3,A_Selected_S003_from_C01921_VF030,True,17,0.709663,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
3,4,A,candidate_pool_random_unit_cell,0.3,0.3,1991,8445,8.483174,5.002873,3.301866,...,-0.027086,0.036474,C03250,4,A_Selected_S004_from_C03250_VF030,True,48,0.612754,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
4,5,A,candidate_pool_random_unit_cell,0.3,0.3,2191,8745,7.982656,4.759386,3.261143,...,-0.114081,0.109409,C03553,5,A_Selected_S005_from_C03553_VF030,True,50,0.821315,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
5,6,A,candidate_pool_random_unit_cell,0.3,0.3,1941,7700,7.934055,5.634739,3.367853,...,0.091142,0.044969,C05371,6,A_Selected_S006_from_C05371_VF030,True,8,0.459395,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
6,7,A,candidate_pool_random_unit_cell,0.3,0.3,2066,8550,8.276864,5.601177,3.455696,...,0.200442,0.009777,C05803,7,A_Selected_S007_from_C05803_VF030,True,33,0.653745,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
7,8,A,candidate_pool_random_unit_cell,0.3,0.3,2266,9450,8.340688,4.546206,3.312869,...,-0.116953,-0.206225,C07153,8,A_Selected_S008_from_C07153_VF030,True,35,0.551047,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
8,9,A,candidate_pool_random_unit_cell,0.3,0.3,2391,9625,8.051025,4.869759,3.161354,...,0.032379,0.125860,C07384,9,A_Selected_S009_from_C07384_VF030,True,19,0.746244,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
9,10,A,candidate_pool_random_unit_cell,0.3,0.3,1891,7900,8.355368,4.697810,3.533489,...,0.001213,0.105232,C08914,10,A_Selected_S010_from_C08914_VF030,True,29,0.430584,11,Global: Strut No. at Nodes-AVG; Global: Strut ...


## 5. 원하는 Candidate/Selected 모델 불러오기 → CAD용 STL/STP Export

후보 long-table 또는 선정본 workbook에서 `Candidate_ID`, `Selected_Model_ID`, `Model_Label`로 원하는 모델만 다시 불러와 CAD 파일을 생성합니다. 기본은 실행되지 않으므로 `MANUAL_EXPORT=True`로 바꿔 실행하세요.


In [5]:
# =============================================================================
# Manual model reload / CAD export cell
# =============================================================================
# 사용법
# 1) 먼저 registry를 확인합니다.
# 2) MODEL_IDS_OR_LABELS에 원하는 Candidate_ID, Selected_Model_ID, 또는 Model_Label을 넣습니다.
# 3) MANUAL_EXPORT=True로 바꾸고 실행하면 선택 모델만 Variables + Summary + STL/STP로 저장됩니다.

RESULT_DIR_TO_LOAD = CFG.output_dir
# 이전 실행 결과 폴더를 불러오려면 아래처럼 직접 지정하세요.
# RESULT_DIR_TO_LOAD = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_153000")

LOAD_SOURCE = "candidate"   # "candidate" or "selected"

# 예시:
# Candidate에서 불러오기: ["C00001", "C01234"] 또는 ["A_Candidate_C00001_VF030"]
# Selected에서 불러오기 : ["1", "2"] 또는 ["A_Selected_S001_from_C00001_VF030"]
MODEL_IDS_OR_LABELS = []

# 먼저 검색/확인용 registry preview
try:
    preview_df = find_models_by_descriptor(
        RESULT_DIR_TO_LOAD,
        source=LOAD_SOURCE,
        query=None,              # 예: '`Target VF` == 0.45'
        sort_by=None,            # 예: 'Global: l/d AVG'
        ascending=True,
        head=30,
    )
    display(preview_df)
except Exception as e:
    print(f"Registry preview skipped: {e}")

# 실제 export 스위치
MANUAL_EXPORT = False

if MANUAL_EXPORT:
    manual_models, manual_summaries, manual_export_dir = export_manual_loaded_models_to_cad(
        result_dir=RESULT_DIR_TO_LOAD,
        ids_or_labels=MODEL_IDS_OR_LABELS,
        source=LOAD_SOURCE,
        cfg=CFG,
        export_stl_flag=True,
        export_stp_flag=True,
    )
    display(pd.DataFrame(manual_summaries))
else:
    print("Manual export is disabled. Set MANUAL_EXPORT=True after filling MODEL_IDS_OR_LABELS.")


,Model_Label,모델명,Candidate_ID,Target VF,Actual VF,Global: Length - AVG,Global: l/d AVG,Global: Angle-Z (weighted with length)-AVG,maxwell_index_M_3D
0,A_Candidate_C00001_VF030,C00001,C00001,0.30,0.30,3.313056,5.202172,29.453563,2008
1,A_Candidate_C00002_VF045,C00002,C00002,0.45,0.45,3.447602,4.855428,30.775912,2558
2,A_Candidate_C00003_VF060,C00003,C00003,0.60,0.60,2.991943,3.696822,30.473134,2308
3,A_Candidate_C00004_VF030,C00004,C00004,0.30,0.30,3.714303,5.991524,27.121522,2008
4,A_Candidate_C00005_VF045,C00005,C00005,0.45,0.45,3.280553,4.299520,30.855444,1833
5,A_Candidate_C00006_VF060,C00006,C00006,0.60,0.60,3.383102,3.935383,33.929764,2208
6,A_Candidate_C00007_VF030,C00007,C00007,0.30,0.30,3.027476,4.948165,31.883781,2083
7,A_Candidate_C00008_VF045,C00008,C00008,0.45,0.45,3.419619,4.638906,34.518802,2283
8,A_Candidate_C00009_VF060,C00009,C00009,0.60,0.60,3.056933,3.726134,24.671801,2558
9,A_Candidate_C00010_VF030,C00010,C00010,0.30,0.30,3.142205,5.345839,30.535703,2183


Manual export is disabled. Set MANUAL_EXPORT=True after filling MODEL_IDS_OR_LABELS.


## 6. Type B 생성 & 구조인자 추출 — 기본값 False, 기존 알고리즘 유지

In [6]:
# =============================================================================
# Type B 생성 & Start/End 기반 구조인자 추출 셀
# - Type B Import 경로의 Variables.xlsx 또는 폴더 내 xlsx 파일을 읽음
# - Variables template 또는 Node-Strut style sheet를 자동 인식
# - 각 sheet를 순차적으로 source lattice로 사용
# - 1-cell이면 5 x 5 x 5로 확장, full-size이면 30 mm로 정규화
# - 목표 VF에 맞춰 radius 역산
# - 최종 Start/End 좌표 table 기준으로 구조인자 추출
# - STL 저장
# =============================================================================

if RUN_TYPE_B:
    type_b_sources = import_type_b_lattices(CFG.type_b_import_path, CFG)
    if not type_b_sources:
        raise RuntimeError(f"No valid Type B sheets were found in: {CFG.type_b_import_path}")

    print(f"Imported Type B source sheets: {len(type_b_sources)}")
    for source_name, source_lat in type_b_sources[:10]:
        print(f"  - {source_name}: nodes={len(source_lat.nodes)}, struts={len(source_lat.edges)}")
    if len(type_b_sources) > 10:
        print(f"  ... and {len(type_b_sources)-10} more sheets")

    new_rows = []
    for vf in CFG.target_vfs:
        for i in range(CFG.n_type_b_per_vf):
            source_name, source_lat = type_b_sources[i % len(type_b_sources)]
            lat = create_type_b_model(float(vf), source_lat, CFG, rng)
            model_id, summary = register_model(lat, CFG)
            new_rows.append(summary)
            print(
                f"[Type B] Model {model_id:>4s} | source={source_name} | "
                f"VF target={vf:.2f}, actual={lat.actual_vf:.4f}, "
                f"nodes={summary['Node 개수']}, struts={summary['Strut 개수']}"
            )

    if SAVE_AFTER_EACH_GENERATION_CELL:
        save_current_outputs(CFG)

    display(pd.DataFrame(new_rows))
else:
    print("Type B generation skipped. Set RUN_TYPE_B = True and check CFG.type_b_import_path to run this cell.")


Type B generation skipped. Set RUN_TYPE_B = True and check CFG.type_b_import_path to run this cell.


## 7. 최종 확인 / Preview

In [7]:
# =============================================================================
# 최종 확인 / Preview
# Type A 실행 후에는 models/summaries에 LHS로 선정된 M개가 들어 있습니다.
# Type B를 별도로 켜서 실행하면 기존 register_model 흐름으로 추가 저장됩니다.
# =============================================================================

if len(models) == 0:
    print("No selected models are currently loaded. Run the Type A or Type B cell first.")
else:
    print(f"Current selected/generated models in runtime: {len(models)}")
    show_summary_tail(min(10, len(summaries)))

    first_model_id = sorted(models.keys(), key=lambda x: int(x))[0]
    print(f"\nPreview: Start/End Variables sheet for model {first_model_id}")
    display(model_variables_df(models[first_model_id]).head(10))

    print("\nMain output folder:", CFG.output_dir)
    if RUN_TYPE_A:
        print("Type A candidate folder:", CFG.output_dir / "TypeA_Candidate_Pool")
        print("Type A selected folder :", CFG.output_dir / "TypeA_Selected_LHS")


Current selected/generated models in runtime: 150


,모델명,Type,Source,Target VF,Actual VF,Node 개수,Strut 개수,Global: Strut No. at Nodes-AVG,Global: Strut No. at Nodes-STDEV,Global: Length - AVG,...,mass_center_y_mm,mass_center_z_mm,Candidate_ID,Selected_Model_ID,Model_Label,LHS_Selected,Selection_Rank,LHS_Distance,LHS_Feature_Count,LHS_Features
140,141,A,candidate_pool_random_unit_cell,0.6,0.6,2266,8900,7.855252,4.784565,3.159852,...,0.004316,-0.064006,C45888,141,A_Selected_S141_from_C45888_VF060,True,25,0.310946,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
141,142,A,candidate_pool_random_unit_cell,0.6,0.6,2116,8325,7.868620,5.153597,3.337062,...,-0.137777,0.123743,C47013,142,A_Selected_S142_from_C47013_VF060,True,19,0.658188,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
142,143,A,candidate_pool_random_unit_cell,0.6,0.6,2066,8500,8.228461,5.461207,3.203373,...,0.223297,0.129495,C49125,143,A_Selected_S143_from_C49125_VF060,True,6,0.999803,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
143,144,A,candidate_pool_random_unit_cell,0.6,0.6,2316,9100,7.858377,4.943637,3.171320,...,-0.044649,-0.353182,C52152,144,A_Selected_S144_from_C52152_VF060,True,3,0.781231,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
144,145,A,candidate_pool_random_unit_cell,0.6,0.6,1741,7325,8.414704,4.669821,3.563074,...,-0.342159,0.128222,C52224,145,A_Selected_S145_from_C52224_VF060,True,30,0.855259,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
145,146,A,candidate_pool_random_unit_cell,0.6,0.6,2116,8675,8.199433,5.428662,3.159413,...,-0.079739,-0.165873,C53106,146,A_Selected_S146_from_C53106_VF060,True,24,0.576090,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
146,147,A,candidate_pool_random_unit_cell,0.6,0.6,2266,9300,8.208297,4.967696,3.330462,...,0.047184,-0.163358,C55467,147,A_Selected_S147_from_C55467_VF060,True,35,0.626509,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
147,148,A,candidate_pool_random_unit_cell,0.6,0.6,2216,9045,8.163357,5.171412,3.220046,...,0.178159,-0.034263,C55572,148,A_Selected_S148_from_C55572_VF060,True,11,0.763649,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
148,149,A,candidate_pool_random_unit_cell,0.6,0.6,2141,9050,8.453993,4.650129,3.180343,...,0.190235,0.037871,C56859,149,A_Selected_S149_from_C56859_VF060,True,29,0.513828,11,Global: Strut No. at Nodes-AVG; Global: Strut ...
149,150,A,candidate_pool_random_unit_cell,0.6,0.6,1966,8150,8.290946,4.798090,3.306394,...,-0.159741,-0.069553,C58455,150,A_Selected_S150_from_C58455_VF060,True,31,0.747383,11,Global: Strut No. at Nodes-AVG; Global: Strut ...



Preview: Start/End Variables sheet for model 1


,Start-x,Start-y,Start-z,Start-Radius,End-x,End-y,End-z,End-radius
0,-15.0,-15.0,-15.0,0.299773,-15.000000,-12.000000,-12.000000,0.299773
1,-15.0,-15.0,-15.0,0.299773,-12.000000,-15.000000,-12.000000,0.299773
2,-15.0,-15.0,-15.0,0.299773,-12.000000,-12.000000,-15.000000,0.299773
3,-15.0,-15.0,-15.0,0.299773,-11.379897,-14.093697,-13.931328,0.299773
4,-15.0,-15.0,-15.0,0.299773,-10.622085,-13.392959,-13.800689,0.299773
5,-15.0,-15.0,-9.0,0.299773,-15.000000,-12.000000,-12.000000,0.299773
6,-15.0,-15.0,-9.0,0.299773,-12.000000,-15.000000,-12.000000,0.299773
7,-15.0,-15.0,-9.0,0.299773,-12.000000,-12.000000,-9.000000,0.299773
8,-15.0,-15.0,-9.0,0.299773,-12.364496,-13.660775,-10.566167,0.299773
9,-15.0,-15.0,-9.0,0.299773,-12.727546,-12.856123,-10.217123,0.299773



Main output folder: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
Type A candidate folder: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\TypeA_Candidate_Pool
Type A selected folder : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\TypeA_Selected_LHS


## 8. Slice image utilities — COM/SOUND, 6-region merge, +5 layer 후처리

이 셀은 lattice `Start/End + radius` 데이터를 직접 rasterize해서 DLP용 slice image를 만듭니다. 첨부한 `6 Regions allocation`의 3×2 병합 방식과 `DLP_Additional_Layer`의 5-layer prepend 방식을 노트북 함수로 통합했습니다.

In [14]:
# =============================================================================
# Slice image utilities: COM/SOUND generation, 6-region merging, +5 layer update
# =============================================================================
# Requirements:
#   pip install pillow
# These utilities rasterize LatticeModel objects directly from Start/End struts.
# They do not require STL slicing software.

import shutil
import traceback
from PIL import Image, ImageDraw


# -----------------------------
# Slice settings
# -----------------------------
@dataclass
class SliceConfig:
    canvas_w: int = 1920
    canvas_h: int = 1080
    total_length_mm: float = 30.0
    layer_height_mm: float = 0.1
    n_com_layers: int = 300
    n_sound_layers: int = 240
    com_period_layers: int = 60
    sound_period_layers: int = 60
    com_repeat_count: int = 5
    sound_repeat_count: int = 4
    use_periodic_layer_copy: bool = True
    pixel_size_mm: float = 0.065
    part_px: int = 461               # 30 mm / 0.065 mm/px ≈ 461.54; user-requested boundary = 461 px
    sound_circle_diameter_px: int = 458
    six_region_crop_w: int = 640
    six_region_crop_h: int = 540
    additional_layers: int = 5
    compression_template_slice: Path = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture\Slice Template\Compression.slice")
    sound_template_slice: Path = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture\Slice Template\Sound.slice")
    verbose: bool = True


SLICE_CFG = SliceConfig(total_length_mm=CFG.total_length_mm)


def _slice_log(msg: str, slice_cfg: SliceConfig = SLICE_CFG) -> None:
    if slice_cfg.verbose:
        print(msg)


def sec_name(layer_no: int) -> str:
    return f"SEC_{int(layer_no):04d}.png"


def natural_sort_key(path_or_name):
    s = Path(path_or_name).name if not isinstance(path_or_name, str) else path_or_name
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]


def list_sec_files(slice_folder: Path) -> Dict[int, Path]:
    out = {}
    for p in Path(slice_folder).glob("SEC_*.png"):
        m = re.fullmatch(r"SEC_(\d+)\.png", p.name, flags=re.IGNORECASE)
        if m:
            out[int(m.group(1))] = p
    return dict(sorted(out.items()))


def count_white_pixels(img_path: Path) -> int:
    with Image.open(img_path) as img:
        gray = img.convert("L")
        return int(np.count_nonzero(np.asarray(gray) > 0))


def _canvas_part_box(slice_cfg: SliceConfig = SLICE_CFG) -> Tuple[int, int, int, int]:
    left = (slice_cfg.canvas_w - slice_cfg.part_px) // 2
    top = (slice_cfg.canvas_h - slice_cfg.part_px) // 2
    return left, top, left + slice_cfg.part_px, top + slice_cfg.part_px


def _node_radii_from_lattice(lat: LatticeModel) -> np.ndarray:
    node_r = np.zeros(len(lat.nodes), dtype=float)
    for (a, b), r in zip(lat.edges, lat.radii):
        node_r[int(a)] = max(node_r[int(a)], float(r))
        node_r[int(b)] = max(node_r[int(b)], float(r))
    return node_r


def _rasterize_lattice_cross_section(
    lat: LatticeModel,
    z_mm: float,
    slice_cfg: SliceConfig = SLICE_CFG,
) -> np.ndarray:
    """Return a 461 x 461 binary image of one z cross-section.

    Pixel scale is 0.065 mm/px. A strut is rasterized by checking the exact
    3D distance from each local pixel center at z=z_mm to each line segment.
    Node spheres are added with their maximum connected strut radius.
    """
    n = int(slice_cfg.part_px)
    px = float(slice_cfg.pixel_size_mm)
    c = (n - 1) / 2.0

    x_coords = (np.arange(n, dtype=float) - c) * px
    y_coords = (c - np.arange(n, dtype=float)) * px
    mask = np.zeros((n, n), dtype=bool)

    nodes = np.asarray(lat.nodes, dtype=float)
    edges = np.asarray(lat.edges, dtype=int)
    radii = np.asarray(lat.radii, dtype=float)

    # Strut capsules/cylinders: exact point-to-segment distance on z plane
    for (a, b), r in zip(edges, radii):
        p0 = nodes[int(a)]
        p1 = nodes[int(b)]
        v = p1 - p0
        vv = float(np.dot(v, v))
        if vv <= 1e-12 or r <= 0:
            continue
        if z_mm < min(p0[2], p1[2]) - r or z_mm > max(p0[2], p1[2]) + r:
            continue

        xmin = min(p0[0], p1[0]) - r
        xmax = max(p0[0], p1[0]) + r
        ymin = min(p0[1], p1[1]) - r
        ymax = max(p0[1], p1[1]) + r

        ix0 = max(0, int(math.floor(xmin / px + c)) - 2)
        ix1 = min(n - 1, int(math.ceil(xmax / px + c)) + 2)
        # y image index is reversed: y = (c - row) * px
        iy0 = max(0, int(math.floor(c - ymax / px)) - 2)
        iy1 = min(n - 1, int(math.ceil(c - ymin / px)) + 2)
        if ix1 < ix0 or iy1 < iy0:
            continue

        xs = x_coords[ix0:ix1 + 1]
        ys = y_coords[iy0:iy1 + 1]
        X, Y = np.meshgrid(xs, ys)
        Z = np.full_like(X, float(z_mm))

        wx = X - p0[0]
        wy = Y - p0[1]
        wz = Z - p0[2]
        t = (wx * v[0] + wy * v[1] + wz * v[2]) / vv
        t = np.clip(t, 0.0, 1.0)
        cxp = p0[0] + t * v[0]
        cyp = p0[1] + t * v[1]
        czp = p0[2] + t * v[2]
        dist2 = (X - cxp) ** 2 + (Y - cyp) ** 2 + (Z - czp) ** 2
        local = dist2 <= float(r) ** 2
        if local.any():
            mask[iy0:iy1 + 1, ix0:ix1 + 1] |= local

    # Node spheres: helps fill intersections for DLP image continuity
    node_r = _node_radii_from_lattice(lat)
    for p, r in zip(nodes, node_r):
        if r <= 0:
            continue
        dz = abs(float(z_mm) - float(p[2]))
        if dz > r:
            continue
        rr = math.sqrt(max(float(r) ** 2 - dz ** 2, 0.0))
        xmin, xmax = p[0] - rr, p[0] + rr
        ymin, ymax = p[1] - rr, p[1] + rr
        ix0 = max(0, int(math.floor(xmin / px + c)) - 2)
        ix1 = min(n - 1, int(math.ceil(xmax / px + c)) + 2)
        iy0 = max(0, int(math.floor(c - ymax / px)) - 2)
        iy1 = min(n - 1, int(math.ceil(c - ymin / px)) + 2)
        if ix1 < ix0 or iy1 < iy0:
            continue
        xs = x_coords[ix0:ix1 + 1]
        ys = y_coords[iy0:iy1 + 1]
        X, Y = np.meshgrid(xs, ys)
        local = (X - p[0]) ** 2 + (Y - p[1]) ** 2 <= rr ** 2
        if local.any():
            mask[iy0:iy1 + 1, ix0:ix1 + 1] |= local

    return (mask.astype(np.uint8) * 255)


def write_lattice_com_slice_folder(
    lat: LatticeModel,
    model_name: str,
    com_root: Path,
    slice_cfg: SliceConfig = SLICE_CFG,
    overwrite: bool = True,
) -> Path:
    """Create one COM-type .slice folder with SEC_0001.png ~ SEC_0300.png.

    Speed optimization:
    - Render only one periodic block (default: 60 layers = 1 cell height for 6 mm / 0.1 mm)
    - Then copy/repeat those images to fill the full 300 layers.
    """
    folder_name = f"{safe_filename(model_name)}.slice"
    out_dir = Path(com_root) / folder_name
    if overwrite and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    left, top, _, _ = _canvas_part_box(slice_cfg)
    z_bottom = -slice_cfg.total_length_mm / 2.0

    n_total = int(slice_cfg.n_com_layers)
    if slice_cfg.use_periodic_layer_copy:
        n_unique = max(1, min(int(slice_cfg.com_period_layers), n_total))
    else:
        n_unique = n_total

    unique_pixel_counts = []
    for layer_no in range(1, n_unique + 1):
        z = z_bottom + (layer_no - 0.5) * slice_cfg.layer_height_mm
        part_img_arr = _rasterize_lattice_cross_section(lat, z, slice_cfg)
        part_img = Image.fromarray(part_img_arr, mode="L")
        canvas = Image.new("L", (slice_cfg.canvas_w, slice_cfg.canvas_h), 0)
        canvas.paste(part_img, (left, top))
        out_path = out_dir / sec_name(layer_no)
        canvas.save(out_path)
        unique_pixel_counts.append(int(np.count_nonzero(part_img_arr > 0)))
        if slice_cfg.verbose and (layer_no % 20 == 0 or layer_no == n_unique):
            print(f"  {folder_name}: rendered unique layer {layer_no}/{n_unique}")

    if slice_cfg.use_periodic_layer_copy and n_total > n_unique:
        for layer_no in range(n_unique + 1, n_total + 1):
            src_layer = ((layer_no - 1) % n_unique) + 1
            shutil.copy2(out_dir / sec_name(src_layer), out_dir / sec_name(layer_no))
            if slice_cfg.verbose and (layer_no % 50 == 0 or layer_no == n_total):
                print(f"  {folder_name}: copied repeated layer {layer_no}/{n_total} (from {src_layer})")

    all_pixel_counts = [unique_pixel_counts[((i - 1) % n_unique)] for i in range(1, n_total + 1)]
    source_period_layers = [((i - 1) % n_unique) + 1 for i in range(1, n_total + 1)]
    generated_mode = ["rendered" if i <= n_unique else "copied" for i in range(1, n_total + 1)]

    pd.DataFrame({
        "Layer": np.arange(1, n_total + 1),
        "File": [sec_name(i) for i in range(1, n_total + 1)],
        "Z_mm_layer_center": [z_bottom + (i - 0.5) * slice_cfg.layer_height_mm for i in range(1, n_total + 1)],
        "WhitePixelCount_in_part_window": all_pixel_counts,
        "Source_Period_Layer": source_period_layers,
        "Generated_Mode": generated_mode,
    }).to_csv(out_dir / "slice_pixel_counts.csv", index=False, encoding="utf-8-sig")

    return out_dir


def generate_com_slices_for_models(
    models_to_slice: Dict[str, LatticeModel],
    com_root: Path,
    slice_cfg: SliceConfig = SLICE_CFG,
    overwrite: bool = True,
) -> pd.DataFrame:
    com_root = Path(com_root)
    com_root.mkdir(parents=True, exist_ok=True)
    manifest = []
    for idx, (model_id, lat) in enumerate(models_to_slice.items(), start=1):
        model_name = safe_filename(model_id)
        _slice_log(f"[COM] {idx}/{len(models_to_slice)} -> {model_name}.slice", slice_cfg)
        folder = write_lattice_com_slice_folder(lat, model_name, com_root, slice_cfg, overwrite=overwrite)
        manifest.append({
            "Model_ID": model_id,
            "Model_Label": getattr(lat, "source_name", ""),
            "Slice_Folder": str(folder),
            "Layer_Count": slice_cfg.n_com_layers,
            "Unique_Rendered_Layers": min(slice_cfg.com_period_layers, slice_cfg.n_com_layers) if slice_cfg.use_periodic_layer_copy else slice_cfg.n_com_layers,
            "Repeat_Count": slice_cfg.com_repeat_count if slice_cfg.use_periodic_layer_copy else 1,
            "Layer_Height_mm": slice_cfg.layer_height_mm,
            "Pixel_Size_mm": slice_cfg.pixel_size_mm,
            "Part_Pixel_Boundary": slice_cfg.part_px,
        })
    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(com_root / "COM_slice_manifest.csv", index=False, encoding="utf-8-sig")
    manifest_df.to_excel(com_root / "COM_slice_manifest.xlsx", index=False)
    return manifest_df


def generate_sound_slices_from_com(
    com_root: Path,
    sound_root: Path,
    slice_cfg: SliceConfig = SLICE_CFG,
    overwrite: bool = True,
) -> pd.DataFrame:
    """Create SOUND-type .slice folders by masking only one periodic 60-layer block,
    then repeating/copying it to 240 layers."""
    com_root = Path(com_root)
    sound_root = Path(sound_root)
    sound_root.mkdir(parents=True, exist_ok=True)
    folders = sorted([p for p in com_root.iterdir() if p.is_dir() and p.suffix.lower() == ".slice"], key=natural_sort_key)

    cx = slice_cfg.canvas_w // 2
    cy = slice_cfg.canvas_h // 2
    d = int(slice_cfg.sound_circle_diameter_px)
    r = d / 2.0
    Y, X = np.ogrid[:slice_cfg.canvas_h, :slice_cfg.canvas_w]
    circle_mask = ((X - cx) ** 2 + (Y - cy) ** 2) <= r ** 2

    n_total = int(slice_cfg.n_sound_layers)
    if slice_cfg.use_periodic_layer_copy:
        n_unique = max(1, min(int(slice_cfg.sound_period_layers), n_total))
    else:
        n_unique = n_total

    manifest = []
    for idx, src_dir in enumerate(folders, start=1):
        dst_dir = sound_root / src_dir.name
        if overwrite and dst_dir.exists():
            shutil.rmtree(dst_dir)
        dst_dir.mkdir(parents=True, exist_ok=True)
        _slice_log(f"[SOUND] {idx}/{len(folders)} -> {dst_dir.name}", slice_cfg)

        for layer_no in range(1, n_unique + 1):
            src_img = src_dir / sec_name(layer_no)
            if not src_img.exists():
                raise FileNotFoundError(f"Missing COM layer: {src_img}")
            with Image.open(src_img) as img:
                arr = np.asarray(img.convert("L"), dtype=np.uint8)
            arr2 = np.where(circle_mask, arr, 0).astype(np.uint8)
            Image.fromarray(arr2, mode="L").save(dst_dir / sec_name(layer_no))
            if slice_cfg.verbose and (layer_no % 20 == 0 or layer_no == n_unique):
                print(f"  {dst_dir.name}: generated unique sound layer {layer_no}/{n_unique}")

        if slice_cfg.use_periodic_layer_copy and n_total > n_unique:
            for layer_no in range(n_unique + 1, n_total + 1):
                src_layer = ((layer_no - 1) % n_unique) + 1
                shutil.copy2(dst_dir / sec_name(src_layer), dst_dir / sec_name(layer_no))
                if slice_cfg.verbose and (layer_no % 40 == 0 or layer_no == n_total):
                    print(f"  {dst_dir.name}: copied repeated sound layer {layer_no}/{n_total} (from {src_layer})")

        manifest.append({
            "Source_COM_Folder": str(src_dir),
            "SOUND_Folder": str(dst_dir),
            "Layer_Count": slice_cfg.n_sound_layers,
            "Unique_Generated_Layers": n_unique,
            "Repeat_Count": slice_cfg.sound_repeat_count if slice_cfg.use_periodic_layer_copy else 1,
            "Circular_Diameter_px": slice_cfg.sound_circle_diameter_px,
        })
    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(sound_root / "SOUND_slice_manifest.csv", index=False, encoding="utf-8-sig")
    manifest_df.to_excel(sound_root / "SOUND_slice_manifest.xlsx", index=False)
    return manifest_df


# -----------------------------
# Template / IDX helpers
# -----------------------------
def resolve_template_files(template_slice_folder: Path) -> Tuple[Path, Optional[Path], Optional[Path]]:
    template_slice_folder = Path(template_slice_folder)
    if not template_slice_folder.exists():
        raise FileNotFoundError(f"Template .slice folder not found: {template_slice_folder}")
    idx_candidates = [template_slice_folder / f"{template_slice_folder.stem}.idx"] + sorted(template_slice_folder.glob("*.idx"))
    idx_candidates = [p for p in idx_candidates if p.exists() and p.is_file()]
    if not idx_candidates:
        raise FileNotFoundError(f"No .idx file found in template folder: {template_slice_folder}")
    idx_template = idx_candidates[0]
    gcode = template_slice_folder / "default.gcode"
    preview = template_slice_folder / "Preview_t.png"
    return idx_template, gcode if gcode.exists() else None, preview if preview.exists() else None


def read_text_auto(path: Path) -> str:
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr", "latin-1"]:
        try:
            return Path(path).read_text(encoding=enc)
        except Exception:
            pass
    return Path(path).read_text(errors="ignore")


def rewrite_idx_text_from_pixel_counts(template_text: str, pixel_counts: Dict[int, int]) -> str:
    """Rewrite TotalLayer, [PixelData], and TotalPixelWhiteCount using actual output images."""
    total_layers = max(pixel_counts.keys()) if pixel_counts else 0
    total_white = int(sum(int(v) for v in pixel_counts.values()))
    text = template_text.replace("\r\n", "\n").replace("\r", "\n")

    text, n1 = re.subn(
        r"(?m)^(\s*TotalLayer\s*=\s*)\d+\s*$",
        rf"\g<1>{total_layers}",
        text,
        count=1,
    )
    if n1 == 0:
        # Insert a minimal BuildData block if the template format is incomplete.
        text = "[BuildData]\nTotalLayer = {0}\n\n".format(total_layers) + text

    pixel_lines = [f"SEC_{i:04d}.png = {int(pixel_counts.get(i, 0))}" for i in range(1, total_layers + 1)]
    new_pixel_block = "\n".join(pixel_lines)

    pattern_pixel_block = re.compile(r"(\[PixelData\]\s*\n)(.*?)(\n\s*\[TotalPixelWhiteCount\])", flags=re.DOTALL | re.IGNORECASE)
    text, n2 = pattern_pixel_block.subn(rf"\1{new_pixel_block}\3", text, count=1)
    if n2 == 0:
        text = text.rstrip() + "\n\n[PixelData]\n" + new_pixel_block + "\n\n[TotalPixelWhiteCount]\nTotalPixelWhiteCount = 0\n"

    text, n3 = re.subn(
        r"(?m)^(\s*TotalPixelWhiteCount\s*=\s*)\d+\s*$",
        rf"\g<1>{total_white}",
        text,
        count=1,
    )
    if n3 == 0:
        text = text.rstrip() + f"\nTotalPixelWhiteCount = {total_white}\n"
    return text


def update_idx_from_images(slice_folder: Path, idx_path: Path, template_text: Optional[str] = None) -> None:
    sec_files = list_sec_files(slice_folder)
    if not sec_files:
        raise RuntimeError(f"No SEC_XXXX.png images found in {slice_folder}")
    pixel_counts = {i: count_white_pixels(p) for i, p in sec_files.items()}
    if template_text is None:
        template_text = read_text_auto(idx_path)
    idx_text = rewrite_idx_text_from_pixel_counts(template_text, pixel_counts)
    Path(idx_path).write_text(idx_text, encoding="utf-8", newline="\n")


def prepend_additional_layers_in_place(
    slice_folder: Path,
    additional_layers: int = 5,
    idx_path: Optional[Path] = None,
) -> None:
    """Prepend N copies of SEC_0001.png and shift existing SEC images by N layers."""
    slice_folder = Path(slice_folder)
    sec_files = list_sec_files(slice_folder)
    if not sec_files:
        raise RuntimeError(f"No SEC files found: {slice_folder}")
    first_file = sec_files[min(sec_files.keys())]
    tmp = slice_folder / "._tmp_additional_layers"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)

    # Additional layers: copies of first layer
    for i in range(1, additional_layers + 1):
        shutil.copy2(first_file, tmp / sec_name(i))

    # Shift original layers
    for old_no, old_path in sec_files.items():
        shutil.copy2(old_path, tmp / sec_name(old_no + additional_layers))

    # Remove old SEC files and move new ones back
    for p in sec_files.values():
        p.unlink()
    for p in sorted(tmp.glob("SEC_*.png"), key=natural_sort_key):
        shutil.move(str(p), str(slice_folder / p.name))
    shutil.rmtree(tmp)

    if idx_path is None:
        candidates = sorted(slice_folder.glob("*.idx"), key=natural_sort_key)
        idx_path = candidates[0] if candidates else None
    if idx_path is not None and Path(idx_path).exists():
        update_idx_from_images(slice_folder, idx_path)


# -----------------------------
# 6-region merge helpers
# -----------------------------
SIX_REGION_POSITIONS = {
    1: (0, 0),
    2: (640, 0),
    3: (1280, 0),
    4: (0, 540),
    5: (640, 540),
    6: (1280, 540),
}


def center_crop_box(img_w: int, img_h: int, crop_w: int, crop_h: int) -> Tuple[int, int, int, int]:
    left = (img_w - crop_w) // 2
    top = (img_h - crop_h) // 2
    return left, top, left + crop_w, top + crop_h


def make_merged_slice_name(batch_folders: List[Path]) -> str:
    stems = [p.stem for p in batch_folders]
    def as_int(s):
        try:
            return int(s)
        except Exception:
            return None
    a = as_int(stems[0])
    b = as_int(stems[-1])
    if a is not None and b is not None:
        return f"{a}-{b}.slice"
    return f"{safe_filename(stems[0])}-{safe_filename(stems[-1])}.slice"


def merge_six_slice_folders(
    slice_folders: List[Path],
    out_folder: Path,
    template_slice_folder: Path,
    slice_cfg: SliceConfig = SLICE_CFG,
    add_layers: bool = True,
    overwrite: bool = True,
) -> Path:
    """Merge up to six .slice folders into one 3 x 2 allocation .slice folder."""
    slice_folders = list(slice_folders)
    if len(slice_folders) == 0:
        raise ValueError("slice_folders is empty")
    if len(slice_folders) > 6:
        raise ValueError("merge_six_slice_folders accepts at most six folders")

    out_folder = Path(out_folder)
    if overwrite and out_folder.exists():
        shutil.rmtree(out_folder)
    out_folder.mkdir(parents=True, exist_ok=True)

    idx_template, gcode_template, preview_template = resolve_template_files(template_slice_folder)
    idx_text = read_text_auto(idx_template)

    if gcode_template is not None:
        shutil.copy2(gcode_template, out_folder / "default.gcode")
    if preview_template is not None:
        shutil.copy2(preview_template, out_folder / "Preview_t.png")

    idx_out = out_folder / f"{out_folder.stem}.idx"
    # Create temporary idx first; update after images are written.
    idx_out.write_text(idx_text, encoding="utf-8", newline="\n")

    sec_maps = [list_sec_files(p) for p in slice_folders]
    total_layers = max(max(m.keys()) for m in sec_maps if m)

    crop_box = center_crop_box(
        slice_cfg.canvas_w,
        slice_cfg.canvas_h,
        slice_cfg.six_region_crop_w,
        slice_cfg.six_region_crop_h,
    )

    for layer_no in range(1, total_layers + 1):
        canvas = Image.new("L", (slice_cfg.canvas_w, slice_cfg.canvas_h), 0)
        for pos, sec_map in enumerate(sec_maps, start=1):
            img_path = sec_map.get(layer_no)
            if img_path is None:
                continue
            with Image.open(img_path) as img:
                patch = img.convert("L").crop(crop_box)
            canvas.paste(patch, SIX_REGION_POSITIONS[pos])
        canvas.save(out_folder / sec_name(layer_no))
        if slice_cfg.verbose and (layer_no % 50 == 0 or layer_no == total_layers):
            print(f"  {out_folder.name}: merged layer {layer_no}/{total_layers}")

    update_idx_from_images(out_folder, idx_out, template_text=idx_text)

    info_lines = [
        f"Merged folder: {out_folder.name}",
        f"Template: {template_slice_folder}",
        f"Additional layers: {slice_cfg.additional_layers if add_layers else 0}",
        "",
        "[Position assignment]",
    ]
    for pos in range(1, 7):
        if pos <= len(slice_folders):
            info_lines.append(f"{pos}: {slice_folders[pos - 1].name}")
        else:
            info_lines.append(f"{pos}: (empty)")
    (out_folder / "combine_info.txt").write_text("\n".join(info_lines), encoding="utf-8")

    if add_layers:
        prepend_additional_layers_in_place(out_folder, additional_layers=slice_cfg.additional_layers, idx_path=idx_out)

    return out_folder


def merge_slice_root_in_batches_of_six(
    source_root: Path,
    merged_root: Path,
    template_slice_folder: Path,
    slice_cfg: SliceConfig = SLICE_CFG,
    add_layers: bool = True,
    overwrite: bool = True,
) -> List[Path]:
    source_root = Path(source_root)
    merged_root = Path(merged_root)
    merged_root.mkdir(parents=True, exist_ok=True)
    folders = sorted([p for p in source_root.iterdir() if p.is_dir() and p.suffix.lower() == ".slice"], key=natural_sort_key)
    if not folders:
        raise RuntimeError(f"No .slice folders found in {source_root}")
    out_folders = []
    for i in range(0, len(folders), 6):
        batch = folders[i:i + 6]
        merged_name = make_merged_slice_name(batch)
        _slice_log(f"[MERGE] {merged_name}: " + ", ".join(p.name for p in batch), slice_cfg)
        out = merge_six_slice_folders(
            slice_folders=batch,
            out_folder=merged_root / merged_name,
            template_slice_folder=template_slice_folder,
            slice_cfg=slice_cfg,
            add_layers=add_layers,
            overwrite=overwrite,
        )
        out_folders.append(out)
    return out_folders


def load_models_for_slicing(
    result_dir: Path,
    source: str = "runtime",
    ids_or_labels: Optional[Sequence[object]] = None,
    cfg: GenerationConfig = CFG,
) -> Dict[str, LatticeModel]:
    """Load models for slicing.

    source='runtime'  : use current `models` dictionary in memory.
    source='selected' : load selected models from TypeA_Selected_StartEnd_Variables.xlsx.
                         If ids_or_labels is empty, load every selected sheet.
    source='candidate': load only requested Candidate_ID/Model_Label values from candidate long-table.
    """
    source = source.lower()
    ids_or_labels = list(ids_or_labels or [])

    if source.startswith("run"):
        if "models" not in globals() or len(models) == 0:
            raise RuntimeError("Runtime `models` is empty. Run Type A/Type B first or use source='selected'.")
        return dict(models)

    if source.startswith("sel"):
        selected_dir = Path(result_dir) / "TypeA_Selected_LHS"
        variables_path = selected_dir / "TypeA_Selected_StartEnd_Variables.xlsx"
        if not variables_path.exists():
            raise FileNotFoundError(f"Selected variables workbook not found: {variables_path}")
        if not ids_or_labels:
            xls = pd.ExcelFile(variables_path)
            ids_or_labels = list(xls.sheet_names)
        return load_type_a_selected_models_from_workbook(result_dir, ids_or_labels, cfg)

    if source.startswith("cand"):
        if not ids_or_labels:
            raise ValueError("For source='candidate', provide Candidate_ID or Model_Label values to avoid slicing all candidates.")
        return load_type_a_candidate_models_from_longtable(result_dir, ids_or_labels, cfg)

    raise ValueError("source must be 'runtime', 'selected', or 'candidate'")

# -----------------------------
# Robust slice-result path helpers (v6)
# -----------------------------
def _has_selected_variables(result_dir: Path) -> bool:
    result_dir = Path(result_dir)
    return (result_dir / "TypeA_Selected_LHS" / "TypeA_Selected_StartEnd_Variables.xlsx").exists()


def find_latest_result_dir_for_slicing(
    base_dir: Optional[Path] = None,
    require_selected: bool = True,
) -> Path:
    """Find the latest Result_* folder that can be used for slicing.

    Priority:
    1) folders under EXPORT_BASE_DIR
    2) folders under current CFG.output_dir.parent
    The returned folder is the most recently modified candidate.
    """
    search_roots = []
    if base_dir is not None:
        search_roots.append(Path(base_dir))
    if "EXPORT_BASE_DIR" in globals():
        search_roots.append(Path(EXPORT_BASE_DIR))
    if "CFG" in globals() and hasattr(CFG, "output_dir"):
        search_roots.append(Path(CFG.output_dir).parent)

    # de-duplicate while preserving order
    unique_roots = []
    seen = set()
    for root in search_roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            unique_roots.append(root)

    candidates = []
    for root in unique_roots:
        if not root.exists():
            continue
        for p in root.glob("Result_*"):
            if not p.is_dir():
                continue
            if require_selected and not _has_selected_variables(p):
                continue
            candidates.append(p)

    if not candidates:
        msg = "No usable Result_* folder found for slicing."
        if require_selected:
            msg += " Expected: Result_*/TypeA_Selected_LHS/TypeA_Selected_StartEnd_Variables.xlsx"
        raise FileNotFoundError(msg)

    return max(candidates, key=lambda p: p.stat().st_mtime)


def resolve_result_dir_for_slicing(user_result_dir: Optional[object] = None) -> Path:
    """Resolve a stable result directory for slicing.

    Use an explicit path when supplied. Otherwise use CFG.output_dir only if it
    already contains TypeA_Selected_LHS/TypeA_Selected_StartEnd_Variables.xlsx.
    If not, fall back to the latest Result_* folder under EXPORT_BASE_DIR.
    """
    if user_result_dir not in [None, "", r""]:
        result_dir = Path(user_result_dir)
        if not result_dir.exists():
            raise FileNotFoundError(f"USER_RESULT_DIR_FOR_SLICING does not exist: {result_dir}")
        return result_dir

    if "CFG" in globals() and hasattr(CFG, "output_dir"):
        cfg_dir = Path(CFG.output_dir)
        if cfg_dir.exists() and _has_selected_variables(cfg_dir):
            return cfg_dir

    return find_latest_result_dir_for_slicing(require_selected=True)


def make_slice_output_paths(result_dir: Path) -> dict:
    result_dir = Path(result_dir)
    slice_output_root = result_dir / "Slice_Output"
    paths = {
        "slice_output_root": slice_output_root,
        "com_root": slice_output_root / "COM" / "Individual",
        "sound_root": slice_output_root / "SOUND" / "Individual",
        "com_merged_root": slice_output_root / "COM" / "Merged",
        "sound_merged_root": slice_output_root / "SOUND" / "Merged",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def infer_slice_model_source(result_dir: Path, requested_source: str = "auto") -> str:
    requested_source = str(requested_source).lower()
    if requested_source != "auto":
        return requested_source
    if _has_selected_variables(result_dir):
        return "selected"
    if "models" in globals() and len(models) > 0:
        return "runtime"
    raise RuntimeError("Cannot infer slicing model source. No selected workbook and runtime `models` is empty.")


def print_slice_tree_status(result_dir: Path) -> None:
    result_dir = Path(result_dir)
    root = result_dir / "Slice_Output"
    print("\n[Slice output status]")
    print("Result dir       :", result_dir)
    print("Slice_Output     :", root, "| exists =", root.exists())
    for sub in ["COM/Individual", "SOUND/Individual", "COM/Merged", "SOUND/Merged"]:
        p = root / sub
        n_folders = len([x for x in p.glob("*.slice") if x.is_dir()]) if p.exists() else 0
        print(f"{sub:16s}: {p} | .slice folders = {n_folders}")



## 9. 이미지 생성 추가셀 1 — COM Type 생성

모델별 `.slice` 폴더를 만들고, 30 mm 높이를 0.1 mm 간격으로 총 300장의 `SEC_0001.png`~`SEC_0300.png` 이미지로 저장합니다. 기본값은 실행되지 않으므로 `RUN_COM_SLICE_GENERATION=True`로 바꿔 실행하세요.

In [15]:
# =============================================================================
# COM Type slice generation — AUTO-RUN v6
# =============================================================================
# 목적:
# - 선택된 lattice 모델을 0.1 mm 간격으로 300장 slicing
# - 1920 x 1080 px 검은 배경 중앙에 461 x 461 px 흰색 단면 배치
# - 저장: Result_*/Slice_Output/COM/Individual/{모델명}.slice/SEC_0001.png ~ SEC_0300.png

# 직접 특정 Result 폴더를 지정하려면 아래에 경로를 입력하세요.
# None이면 CFG.output_dir가 유효할 때 사용하고, 아니면 EXPORT_BASE_DIR 아래 최신 Result_* 폴더를 자동 탐색합니다.
USER_RESULT_DIR_FOR_SLICING = None
# 예시:
# USER_RESULT_DIR_FOR_SLICING = r"C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242"

RUN_COM_SLICE_GENERATION = True
SLICE_MODEL_SOURCE = "auto"          # "auto", "selected", "runtime", or "candidate"
SLICE_MODEL_IDS_OR_LABELS = []        # selected에서 비우면 전체 selected sheet 로드
SLICE_OVERWRITE_INDIVIDUAL = True

# 주기구조 기반 속도 개선 설정
SLICE_CFG.use_periodic_layer_copy = True
SLICE_CFG.com_period_layers = 60
SLICE_CFG.com_repeat_count = 5
assert SLICE_CFG.com_period_layers * SLICE_CFG.com_repeat_count == SLICE_CFG.n_com_layers, \
    "COM periodic setting must satisfy period_layers * repeat_count == n_com_layers"

RESULT_DIR_FOR_SLICING = resolve_result_dir_for_slicing(USER_RESULT_DIR_FOR_SLICING)
SLICE_MODEL_SOURCE_RESOLVED = infer_slice_model_source(RESULT_DIR_FOR_SLICING, SLICE_MODEL_SOURCE)

_paths = make_slice_output_paths(RESULT_DIR_FOR_SLICING)
SLICE_OUTPUT_ROOT = _paths["slice_output_root"]
COM_ROOT = _paths["com_root"]
SOUND_ROOT = _paths["sound_root"]
COM_MERGED_ROOT = _paths["com_merged_root"]
SOUND_MERGED_ROOT = _paths["sound_merged_root"]

print("[COM slicing settings]")
print("Result dir       :", RESULT_DIR_FOR_SLICING)
print("Model source     :", SLICE_MODEL_SOURCE_RESOLVED)
print("COM output root  :", COM_ROOT)
print("Layer count      :", SLICE_CFG.n_com_layers)
print("Unique render    :", SLICE_CFG.com_period_layers, "layers")
print("Repeat count     :", SLICE_CFG.com_repeat_count, "times")
print("Layer height     :", SLICE_CFG.layer_height_mm, "mm")
print("Pixel size       :", SLICE_CFG.pixel_size_mm, "mm/px")

if RUN_COM_SLICE_GENERATION:
    slice_models = load_models_for_slicing(
        result_dir=RESULT_DIR_FOR_SLICING,
        source=SLICE_MODEL_SOURCE_RESOLVED,
        ids_or_labels=SLICE_MODEL_IDS_OR_LABELS,
        cfg=CFG,
    )
    if len(slice_models) == 0:
        raise RuntimeError("No model was loaded for COM slicing.")

    print(f"Loaded {len(slice_models)} model(s) for COM slicing.")
    com_manifest_df = generate_com_slices_for_models(
        models_to_slice=slice_models,
        com_root=COM_ROOT,
        slice_cfg=SLICE_CFG,
        overwrite=SLICE_OVERWRITE_INDIVIDUAL,
    )
    display(com_manifest_df)
    print("\n[COM slicing finished]")
    print("COM slice folder:", COM_ROOT)
    print_slice_tree_status(RESULT_DIR_FOR_SLICING)
else:
    print("COM slice generation was skipped because RUN_COM_SLICE_GENERATION=False.")
    print_slice_tree_status(RESULT_DIR_FOR_SLICING)


[COM slicing settings]
Result dir       : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
Model source     : selected
COM output root  : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Individual
Layer count      : 300
Unique render    : 60 layers
Repeat count     : 5 times
Layer height     : 0.1 mm
Pixel size       : 0.065 mm/px
Loaded 150 model(s) for COM slicing.
[COM] 1/150 -> 1.slice
  1.slice: rendered unique layer 20/60
  1.slice: rendered unique layer 40/60
  1.slice: rendered unique layer 60/60
  1.slice: copied repeated layer 100/300 (from 40)
  1.slice: copied repeated layer 150/300 (from 30)
  1.slice: copied repeated layer 200/300 (from 20)
  1.slice: copied repeated layer 250/300 (from 10)
  1.slice: copied repeated layer 300/300 (from 60)
[COM] 2/150 -> 2.slice
  2.slice: rendered unique layer 20/60
  2.slice: rendered unique layer 40/60
  2.slice: rendered unique layer 60/60
  2.slice: copied repeated layer 100/30

,Model_ID,Model_Label,Slice_Folder,Layer_Count,Unique_Rendered_Layers,Repeat_Count,Layer_Height_mm,Pixel_Size_mm,Part_Pixel_Boundary
0,1,selected_1,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
1,2,selected_2,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
2,3,selected_3,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
3,4,selected_4,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
4,5,selected_5,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
...,...,...,...,...,...,...,...,...,...
145,146,selected_146,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
146,147,selected_147,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
147,148,selected_148,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461
148,149,selected_149,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,300,60,5,0.1,0.065,461



[COM slicing finished]
COM slice folder: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Individual

[Slice output status]
Result dir       : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
Slice_Output     : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output | exists = True
COM/Individual  : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Individual | .slice folders = 150
SOUND/Individual: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Individual | .slice folders = 0
COM/Merged      : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Merged | .slice folders = 0
SOUND/Merged    : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Merged | .slice folders = 0


## 10. 이미지 생성 추가셀 2 — SOUND Type 생성

COM에서 생성한 1920×1080 이미지를 기준으로, 중앙 458 px 지름 원형 영역만 남기고 `SEC_0001.png`~`SEC_0240.png` 240장만 SOUND `.slice` 폴더로 저장합니다.

In [16]:
# =============================================================================
# SOUND Type slice generation — AUTO-RUN v6
# =============================================================================
# 목적:
# - COM/Individual/*.slice 폴더를 기반으로 SOUND/Individual/*.slice 생성
# - 중앙 기준 지름 458 px 원형으로 crop/mask
# - SEC_0001.png ~ SEC_0240.png, 총 240장 저장

RUN_SOUND_SLICE_GENERATION = True
SOUND_OVERWRITE_INDIVIDUAL = True

# 주기구조 기반 속도 개선 설정
SLICE_CFG.use_periodic_layer_copy = True
SLICE_CFG.sound_period_layers = 60
SLICE_CFG.sound_repeat_count = 4
assert SLICE_CFG.sound_period_layers * SLICE_CFG.sound_repeat_count == SLICE_CFG.n_sound_layers, \
    "SOUND periodic setting must satisfy period_layers * repeat_count == n_sound_layers"

# 앞 COM 셀을 건너뛰고 이 셀만 실행해도 경로가 잡히도록 보정
if "RESULT_DIR_FOR_SLICING" not in globals():
    USER_RESULT_DIR_FOR_SLICING = None
    RESULT_DIR_FOR_SLICING = resolve_result_dir_for_slicing(USER_RESULT_DIR_FOR_SLICING)
    _paths = make_slice_output_paths(RESULT_DIR_FOR_SLICING)
    SLICE_OUTPUT_ROOT = _paths["slice_output_root"]
    COM_ROOT = _paths["com_root"]
    SOUND_ROOT = _paths["sound_root"]
    COM_MERGED_ROOT = _paths["com_merged_root"]
    SOUND_MERGED_ROOT = _paths["sound_merged_root"]

print("[SOUND slicing settings]")
print("Result dir        :", RESULT_DIR_FOR_SLICING)
print("Input COM root    :", COM_ROOT)
print("SOUND output root :", SOUND_ROOT)
print("Layer count       :", SLICE_CFG.n_sound_layers)
print("Unique generated  :", SLICE_CFG.sound_period_layers, "layers")
print("Repeat count      :", SLICE_CFG.sound_repeat_count, "times")
print("Circle diameter   :", SLICE_CFG.sound_circle_diameter_px, "px")

if RUN_SOUND_SLICE_GENERATION:
    com_folders = sorted([p for p in Path(COM_ROOT).glob("*.slice") if p.is_dir()], key=natural_sort_key)
    if len(com_folders) == 0:
        raise RuntimeError(
            "No COM .slice folders found. Run the COM Type slice generation cell first.\n"
            f"Expected folder: {COM_ROOT}"
        )
    print(f"Found {len(com_folders)} COM .slice folder(s).")

    sound_manifest_df = generate_sound_slices_from_com(
        com_root=COM_ROOT,
        sound_root=SOUND_ROOT,
        slice_cfg=SLICE_CFG,
        overwrite=SOUND_OVERWRITE_INDIVIDUAL,
    )
    display(sound_manifest_df)
    print("\n[SOUND slicing finished]")
    print("SOUND slice folder:", SOUND_ROOT)
    print_slice_tree_status(RESULT_DIR_FOR_SLICING)
else:
    print("SOUND slice generation was skipped because RUN_SOUND_SLICE_GENERATION=False.")
    print_slice_tree_status(RESULT_DIR_FOR_SLICING)


[SOUND slicing settings]
Result dir        : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
Input COM root    : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Individual
SOUND output root : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Individual
Layer count       : 240
Unique generated  : 60 layers
Repeat count      : 4 times
Circle diameter   : 458 px
Found 150 COM .slice folder(s).
[SOUND] 1/150 -> 1.slice
  1.slice: generated unique sound layer 20/60
  1.slice: generated unique sound layer 40/60
  1.slice: generated unique sound layer 60/60
  1.slice: copied repeated sound layer 80/240 (from 20)
  1.slice: copied repeated sound layer 120/240 (from 60)
  1.slice: copied repeated sound layer 160/240 (from 40)
  1.slice: copied repeated sound layer 200/240 (from 20)
  1.slice: copied repeated sound layer 240/240 (from 60)
[SOUND] 2/150 -> 2.slice
  2.slice: generated unique sound layer 

,Source_COM_Folder,SOUND_Folder,Layer_Count,Unique_Generated_Layers,Repeat_Count,Circular_Diameter_px
0,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
1,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
2,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
3,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
4,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
...,...,...,...,...,...,...
145,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
146,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
147,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458
148,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,C:\Users\김민겸\Desktop\AI-lattice architecture\R...,240,60,4,458



[SOUND slicing finished]
SOUND slice folder: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Individual

[Slice output status]
Result dir       : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
Slice_Output     : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output | exists = True
COM/Individual  : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Individual | .slice folders = 150
SOUND/Individual: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Individual | .slice folders = 150
COM/Merged      : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Merged | .slice folders = 0
SOUND/Merged    : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Merged | .slice folders = 0


## 11. 이미지 후처리 — COM & SOUND 6개 병합 + template 복사 + 5 layer 생성

각 타입의 individual `.slice` 폴더를 이름순으로 6개씩 묶어 3×2 위치에 병합합니다. 병합 폴더명은 `1-6.slice`처럼 첫 모델명과 마지막 모델명을 합쳐 생성합니다. 이후 template의 `.idx`, `default.gcode`, `Preview_t.png`를 복사하고, `.idx` 이름은 병합 폴더명과 동일하게 바꾸며, 마지막으로 앞쪽에 5개 추가 layer를 생성합니다.

In [17]:
# =============================================================================
# COM & SOUND postprocess — AUTO-RUN v6
# 6-region merge + template copy + 5 additional layers
# =============================================================================
# 출력:
# - Result_*/Slice_Output/COM/Merged/{1-6}.slice
# - Result_*/Slice_Output/SOUND/Merged/{1-6}.slice
# 각 병합 폴더에는 SEC 이미지, {폴더명}.idx, default.gcode, Preview_t.png가 저장됩니다.

RUN_COM_MERGE_AND_ADD_LAYERS = True
RUN_SOUND_MERGE_AND_ADD_LAYERS = True
MERGE_OVERWRITE = True

# Template paths can be edited here if needed.
SLICE_CFG.compression_template_slice = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture\Slice Template\Compression.slice")
SLICE_CFG.sound_template_slice = Path(r"C:\Users\김민겸\Desktop\AI-lattice architecture\Slice Template\Sound.slice")
SLICE_CFG.additional_layers = 5

# 이 셀만 단독 실행해도 경로가 잡히도록 보정
if "RESULT_DIR_FOR_SLICING" not in globals():
    USER_RESULT_DIR_FOR_SLICING = None
    RESULT_DIR_FOR_SLICING = resolve_result_dir_for_slicing(USER_RESULT_DIR_FOR_SLICING)
    _paths = make_slice_output_paths(RESULT_DIR_FOR_SLICING)
    SLICE_OUTPUT_ROOT = _paths["slice_output_root"]
    COM_ROOT = _paths["com_root"]
    SOUND_ROOT = _paths["sound_root"]
    COM_MERGED_ROOT = _paths["com_merged_root"]
    SOUND_MERGED_ROOT = _paths["sound_merged_root"]

print("[Merge/add-layer settings]")
print("Result dir          :", RESULT_DIR_FOR_SLICING)
print("COM individual root :", COM_ROOT)
print("SOUND individual root:", SOUND_ROOT)
print("COM merged root     :", COM_MERGED_ROOT)
print("SOUND merged root   :", SOUND_MERGED_ROOT)
print("COM template        :", SLICE_CFG.compression_template_slice)
print("SOUND template      :", SLICE_CFG.sound_template_slice)
print("Additional layers   :", SLICE_CFG.additional_layers)

if RUN_COM_MERGE_AND_ADD_LAYERS:
    com_input_folders = sorted([p for p in Path(COM_ROOT).glob("*.slice") if p.is_dir()], key=natural_sort_key)
    if len(com_input_folders) == 0:
        raise RuntimeError(
            "No COM individual .slice folders found. Run COM Type slice generation first.\n"
            f"Expected folder: {COM_ROOT}"
        )
    if not Path(SLICE_CFG.compression_template_slice).exists():
        raise FileNotFoundError(f"Compression template folder not found: {SLICE_CFG.compression_template_slice}")

    com_merged_folders = merge_slice_root_in_batches_of_six(
        source_root=COM_ROOT,
        merged_root=COM_MERGED_ROOT,
        template_slice_folder=SLICE_CFG.compression_template_slice,
        slice_cfg=SLICE_CFG,
        add_layers=True,
        overwrite=MERGE_OVERWRITE,
    )
    print(f"\nCOM merged folders created: {len(com_merged_folders)}")
    for p in com_merged_folders:
        print(" -", p)
else:
    print("COM merge/add-layer was skipped because RUN_COM_MERGE_AND_ADD_LAYERS=False.")

if RUN_SOUND_MERGE_AND_ADD_LAYERS:
    sound_input_folders = sorted([p for p in Path(SOUND_ROOT).glob("*.slice") if p.is_dir()], key=natural_sort_key)
    if len(sound_input_folders) == 0:
        raise RuntimeError(
            "No SOUND individual .slice folders found. Run SOUND Type slice generation first.\n"
            f"Expected folder: {SOUND_ROOT}"
        )
    if not Path(SLICE_CFG.sound_template_slice).exists():
        raise FileNotFoundError(f"Sound template folder not found: {SLICE_CFG.sound_template_slice}")

    sound_merged_folders = merge_slice_root_in_batches_of_six(
        source_root=SOUND_ROOT,
        merged_root=SOUND_MERGED_ROOT,
        template_slice_folder=SLICE_CFG.sound_template_slice,
        slice_cfg=SLICE_CFG,
        add_layers=True,
        overwrite=MERGE_OVERWRITE,
    )
    print(f"\nSOUND merged folders created: {len(sound_merged_folders)}")
    for p in sound_merged_folders:
        print(" -", p)
else:
    print("SOUND merge/add-layer was skipped because RUN_SOUND_MERGE_AND_ADD_LAYERS=False.")

print_slice_tree_status(RESULT_DIR_FOR_SLICING)
print("\nAll slice post-processing finished.")


[Merge/add-layer settings]
Result dir          : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242
COM individual root : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Individual
SOUND individual root: C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Individual
COM merged root     : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\COM\Merged
SOUND merged root   : C:\Users\김민겸\Desktop\AI-lattice architecture\Result_20260707_031242\Slice_Output\SOUND\Merged
COM template        : C:\Users\김민겸\Desktop\AI-lattice architecture\Slice Template\Compression.slice
SOUND template      : C:\Users\김민겸\Desktop\AI-lattice architecture\Slice Template\Sound.slice
Additional layers   : 5
[MERGE] 1-6.slice: 1.slice, 2.slice, 3.slice, 4.slice, 5.slice, 6.slice
  1-6.slice: merged layer 50/300
  1-6.slice: merged layer 100/300
  1-6.slice: merged layer 150/300
  1-6.slice: merged lay